# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%additional_python_modules matplotlib,seaborn,numpy,lightgbm,scikit-learn



import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Additional python modules to be included:
matplotlib
seaborn
numpy
lightgbm
scikit-learn
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 2880
Session ID: 46d2f0ff-7713-4133-abda-08d84c770703
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
--additional-python-modules matplotlib,se

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [2]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql.types import DoubleType, IntegerType, StructType, StructField

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Step 1: Load data from S3
# ------------------------------------------------------------
TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
TEST_PATH  = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_test.csv"
OUTPUT_PATH = "s3://test-magangal/submission/"


In [3]:

# Read with Spark, then convert to pandas because:
# - Dataset is small (~5000 rows), fits easily in memory
# - sklearn/lightgbm have way more model options than Spark MLlib
train_df = spark.read.csv(TRAIN_PATH, header=True, inferSchema=True).toPandas()
test_df  = spark.read.csv(TEST_PATH,  header=True, inferSchema=True).toPandas()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTarget statistics:")
print(train_df["target"].describe())

# ------------------------------------------------------------
# Step 2: Quick look at the target
# ------------------------------------------------------------
# I noticed the target has some HUGE values (positive and negative).
# This will mess up regular MSE loss because outliers dominate.
# Let me check the percentiles to confirm:
print("\nTarget percentiles:")
print(train_df["target"].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))


Train shape: (2500, 17)
Test shape: (2500, 16)

Target statistics:
count     2500.000000
mean       -22.043467
std       1978.177302
min     -41008.015927
25%        -43.166743
50%         -0.769750
75%         42.000951
max      69628.199874
Name: target, dtype: float64

Target percentiles:
0.01   -2654.943612
0.05    -127.632454
0.25     -43.166743
0.50      -0.769750
0.75      42.000951
0.95     120.914947
0.99    1591.055867
Name: target, dtype: float64


In [4]:

# Yeah, 99% of targets are within reasonable range but a few are extreme.
# This means I should either:
#  (a) use a model robust to outliers (Huber loss)
#  (b) use trees, which are less affected by outlier targets than linear models
# I'll do both and compare.

# ------------------------------------------------------------
# Step 3: Split features and target
# ------------------------------------------------------------
feature_cols = [f"x{i}" for i in range(15)]

X_train = train_df[feature_cols].copy()
y_train = train_df["target"].copy()

X_test = test_df[feature_cols].copy()
test_ids = test_df["Id"].values

print("\nMissing values per column (train):")
print(X_train.isna().sum())



Missing values per column (train):
x0     32
x1     30
x2     22
x3     24
x4     34
x5     28
x6     30
x7     30
x8     34
x9     30
x10    31
x11    32
x12    38
x13    28
x14    25
dtype: int64


In [5]:

# ------------------------------------------------------------
# Step 4: Set up cross-validation
# ------------------------------------------------------------
# I'll use 5-fold CV to estimate how each model performs.
# R² is the metric the competition uses.

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cross_validate(model_builder, X, y, X_test_eval, name):
    """
    Run 5-fold CV. For each fold:
      - Train model on 4 folds
      - Predict on the 5th (out-of-fold predictions)
      - Also predict on test set, average across folds
    Returns OOF predictions, averaged test predictions, and R² score.
    """
    oof = np.zeros(len(X))
    test_preds = np.zeros(len(X_test_eval))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        model = model_builder()
        
        # Get train/val splits (handle both pandas and numpy)
        if isinstance(X, pd.DataFrame):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        else:
            X_tr, X_val = X[train_idx], X[val_idx]
        y_tr = y.iloc[train_idx] if isinstance(y, pd.Series) else y[train_idx]
        
        model.fit(X_tr, y_tr)
        oof[val_idx] = model.predict(X_val)
        test_preds += model.predict(X_test_eval) / kf.get_n_splits()
    
    score = r2_score(y, oof)
    print(f"  {name}: R² = {score:.4f}")
    return oof, test_preds, score



In [6]:

# ------------------------------------------------------------
# Step 5: Model 1 — Ridge regression (baseline)
# ------------------------------------------------------------
# Always start with a simple baseline. Ridge handles correlated features
# better than plain linear regression.

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

print("\n--- Training Ridge (baseline) ---")

def make_ridge():
    # Pipeline: fill NaNs with median, scale, then Ridge
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
        ("model",  Ridge(alpha=1.0)),
    ])

oof_ridge, test_ridge, r2_ridge = cross_validate(
    make_ridge, X_train, y_train, X_test, "Ridge"
)



--- Training Ridge (baseline) ---
  Ridge: R² = 0.0330


In [7]:

# ------------------------------------------------------------
# Step 6: Model 2 — Huber regression (robust to outliers)
# ------------------------------------------------------------
# Since target has outliers, Huber loss should do better than MSE.
# Huber is like MSE for small errors but linear for big ones.

from sklearn.linear_model import HuberRegressor

print("\n--- Training Huber (robust linear) ---")

def make_huber():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
        ("model",  HuberRegressor(epsilon=1.35, max_iter=500, alpha=0.001)),
    ])

oof_huber, test_huber, r2_huber = cross_validate(
    make_huber, X_train, y_train, X_test, "Huber"
)



--- Training Huber (robust linear) ---
  Huber: R² = 0.0030


In [8]:

# ------------------------------------------------------------
# Step 7: Model 3 — LightGBM (usually wins on tabular data)
# ------------------------------------------------------------
# Trees handle:
#  - Non-linear relationships (good if features interact)
#  - Missing values natively (no need to impute!)
#  - Outliers in features
# I'm using huber objective so it's also robust to target outliers.

print("\n--- Training LightGBM ---")

try:
    import lightgbm as lgb
    
    def make_lgb():
        return lgb.LGBMRegressor(
            objective="huber",       # robust loss
            n_estimators=2000,
            learning_rate=0.02,
            num_leaves=31,
            min_child_samples=20,    # prevent overfitting
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.5,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )
    
    # Note: passing X_train directly (no imputation) — LightGBM handles NaN
    oof_lgb, test_lgb, r2_lgb = cross_validate(
        make_lgb, X_train, y_train, X_test, "LightGBM"
    )
    have_lgb = True
except ImportError:
    print("  LightGBM not installed, skipping")
    have_lgb = False
    r2_lgb = -999



--- Training LightGBM ---
  LightGBM: R² = 0.0008


In [9]:

# ------------------------------------------------------------
# Step 8: Pick the best model (or blend)
# ------------------------------------------------------------
# Compare the OOF R² scores. Whichever is best on OOF is probably
# best on the test set too.

print("\n=== Summary ===")
print(f"Ridge:    R² = {r2_ridge:.4f}")
print(f"Huber:    R² = {r2_huber:.4f}")
if have_lgb:
    print(f"LightGBM: R² = {r2_lgb:.4f}")

# Try a simple average of the models too — sometimes blending helps
# because different models make different mistakes.
if have_lgb:
    # Average all three
    oof_blend = (oof_ridge + oof_huber + oof_lgb) / 3
    test_blend = (test_ridge + test_huber + test_lgb) / 3
    r2_blend = r2_score(y_train, oof_blend)
    print(f"Blend:    R² = {r2_blend:.4f}")
else:
    oof_blend = (oof_ridge + oof_huber) / 2
    test_blend = (test_ridge + test_huber) / 2
    r2_blend = r2_score(y_train, oof_blend)
    print(f"Blend:    R² = {r2_blend:.4f}")

# Pick the best one
options = [
    ("Ridge", test_ridge, r2_ridge),
    ("Huber", test_huber, r2_huber),
    ("Blend", test_blend, r2_blend),
]
if have_lgb:
    options.append(("LightGBM", test_lgb, r2_lgb))

best_name, best_preds, best_score = max(options, key=lambda x: x[2])
print(f"\nBest model: {best_name} (R² = {best_score:.4f})")



=== Summary ===
Ridge:    R² = 0.0330
Huber:    R² = 0.0030
LightGBM: R² = 0.0008
Blend:    R² = 0.0261

Best model: Ridge (R² = 0.0330)


In [10]:

# ------------------------------------------------------------
# Step 9: Make submission file
# ------------------------------------------------------------
submission = pd.DataFrame({
    "Id": test_ids,
    "target": best_preds
})

print("\nSubmission preview:")
print(submission.head())
print("Submission shape:", submission.shape)

# Save locally so we can download it
submission.to_csv("/tmp/submission.csv", index=False)

# Also write back to S3 via Spark
schema = StructType([
    StructField("Id", IntegerType(), True),
    StructField("target", DoubleType(), True),
])
sub_spark = spark.createDataFrame(submission, schema=schema)
(sub_spark
    .coalesce(1)              # so we get a single CSV file
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(OUTPUT_PATH))

print(f"\nDone! Submission saved to {OUTPUT_PATH}")
print("Also saved locally to /tmp/submission.csv")


Submission preview:
   Id      target
0   0 -546.647944
1   6 -425.784724
2   7    5.891397
3   8 -548.386688
4  12  667.052447
Submission shape: (2500, 2)

Done! Submission saved to s3://test-magangal/submission/
Also saved locally to /tmp/submission.csv


In [12]:
import pandas as pd
import numpy as np

train_df = pd.read_csv(TRAIN_PATH)
feature_cols = [f"x{i}" for i in range(15)]
X = train_df[feature_cols].copy()
y = train_df["target"].copy()

# 1. Check target distribution
print("Target stats:")
print(y.describe())
print(f"\nValues > 1000:  {(y.abs() > 1000).sum()}")
print(f"Values > 5000:  {(y.abs() > 5000).sum()}")
print(f"Values > 10000: {(y.abs() > 10000).sum()}")

# 2. Linear correlations
X_filled = X.fillna(X.median())
print("\nLinear corr with target:")
corrs = []
for c in feature_cols:
    corr = np.corrcoef(X_filled[c], y)[0, 1]
    corrs.append((c, corr))
    print(f"  {c}: {corr:+.4f}")

# 3. Check if PRODUCTS of features correlate strongly
print("\nTop pairwise products vs target:")
products = []
for i in range(15):
    for j in range(i, 15):  # include squares (i==j)
        prod = X_filled[f"x{i}"] * X_filled[f"x{j}"]
        corr = np.corrcoef(prod, y)[0, 1]
        products.append((f"x{i}*x{j}", corr))
products.sort(key=lambda t: -abs(t[1]))
for name, corr in products[:15]:
    print(f"  {name}: {corr:+.4f}")

# 4. Check if TRIPLE products help
print("\nTop triple products vs target:")
triples = []
for i in range(15):
    for j in range(i, 15):
        for k in range(j, 15):
            prod = X_filled[f"x{i}"] * X_filled[f"x{j}"] * X_filled[f"x{k}"]
            corr = np.corrcoef(prod, y)[0, 1]
            triples.append((f"x{i}*x{j}*x{k}", corr))
triples.sort(key=lambda t: -abs(t[1]))
for name, corr in triples[:15]:
    print(f"  {name}: {corr:+.4f}")

Target stats:
count     2500.000000
mean       -22.043467
std       1978.177302
min     -41008.015927
25%        -43.166743
50%         -0.769750
75%         42.000951
max      69628.199874
Name: target, dtype: float64

Values > 1000:  84
Values > 5000:  24
Values > 10000: 10

Linear corr with target:
  x0: +0.0033
  x1: +0.0049
  x2: -0.0227
  x3: -0.0020
  x4: +0.0283
  x5: -0.0262
  x6: -0.0404
  x7: +0.0204
  x8: -0.0475
  x9: +0.2319
  x10: -0.0234
  x11: +0.0274
  x12: +0.0149
  x13: +0.0029
  x14: -0.0031

Top pairwise products vs target:
  x9*x11: +0.1599
  x9*x12: +0.1053
  x2*x9: -0.1052
  x0*x9: -0.0917
  x8*x9: -0.0898
  x5*x9: +0.0889
  x1*x9: +0.0849
  x11*x11: +0.0753
  x7*x11: +0.0664
  x1*x11: +0.0543
  x1*x7: +0.0506
  x9*x10: -0.0489
  x9*x9: +0.0486
  x3*x9: -0.0476
  x11*x12: +0.0468

Top triple products vs target:
  x9*x9*x9: +0.4517
  x9*x11*x11: +0.3662
  x9*x9*x11: +0.3442
  x2*x9*x11: -0.3208
  x8*x9*x9: -0.2889
  x7*x9*x11: +0.2878
  x8*x9*x11: -0.2515
  x7*x

In [6]:
# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------
print("Loading data from S3...")
train_df = spark.read.csv(TRAIN_PATH, header=True, inferSchema=True).toPandas()
test_df  = spark.read.csv(TEST_PATH, header=True, inferSchema=True).toPandas()

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

feature_cols = [f"x{i}" for i in range(15)]
X = train_df[feature_cols].copy()
y = train_df["target"].copy()
X_test = test_df[feature_cols].copy()
test_ids = test_df["Id"].values

# Impute missing values
imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=feature_cols)


Loading data from S3...
Train shape: (2500, 17)
Test shape:  (2500, 16)


In [7]:

# ------------------------------------------------------------
# Signed log transform (compresses outliers, preserves sign)
# ------------------------------------------------------------
def signed_log(arr):
    return np.sign(arr) * np.log1p(np.abs(arr))

def signed_log_inverse(arr):
    return np.sign(arr) * np.expm1(np.abs(arr))

print(f"\nTarget stats (raw):")
print(f"  std={y.std():.0f}, min={y.min():.0f}, max={y.max():.0f}")

y_log = signed_log(y)
print(f"Target stats (after signed-log):")
print(f"  std={y_log.std():.2f}, min={y_log.min():.2f}, max={y_log.max():.2f}")

# ------------------------------------------------------------
# CV helper that applies transform internally
# ------------------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_with_log(model_fn, X, y_orig, X_te, name):
    """Train on signed_log(y), score on original y."""
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_te))
    y_train_log = signed_log(y_orig)
    
    for tr, va in kf.split(X):
        m = model_fn()
        Xtr = X.iloc[tr] if hasattr(X, 'iloc') else X[tr]
        Xva = X.iloc[va] if hasattr(X, 'iloc') else X[va]
        ytr = y_train_log.iloc[tr] if hasattr(y_train_log, 'iloc') else y_train_log[tr]
        m.fit(Xtr, ytr)
        # Predict in log-space, then transform back
        oof[va] = signed_log_inverse(m.predict(Xva))
        test_pred += signed_log_inverse(m.predict(X_te)) / kf.n_splits
    
    score = r2_score(y_orig, oof)
    print(f"  {name}: R² = {score:.4f}")
    return oof, test_pred, score


# ============================================================
# MODEL 1: Polynomial deg-3 + Ridge (with log target)
# ============================================================
print("\n=== Model 1: Poly3 + Ridge (log-transformed target) ===")

best_poly_score = -np.inf
best_poly_test = None
best_poly_oof = None
best_alpha = None

for alpha in [0.1, 1, 10, 100, 1000]:
    def make_model(a=alpha):
        return Pipeline([
            ("poly", PolynomialFeatures(degree=3, include_bias=False)),
            ("scale", StandardScaler()),
            ("ridge", Ridge(alpha=a)),
        ])
    oof, test_pred, r2 = cv_with_log(make_model, X_imp, y, X_test_imp, f"alpha={alpha}")
    if r2 > best_poly_score:
        best_poly_score = r2
        best_poly_test = test_pred
        best_poly_oof = oof
        best_alpha = alpha

print(f"\nBest Poly3+Ridge: alpha={best_alpha}, R²={best_poly_score:.4f}")

# ============================================================
# MODEL 2: Targeted features (focused on x9, x11) + Ridge
# ============================================================
print("\n=== Model 2: Targeted features + Ridge ===")

def make_targeted_features(df):
    """Build features all at once for performance."""
    new_cols = {}
    
    # Powers of x9 and x11 up to degree 4
    for d1 in range(5):
        for d2 in range(5):
            if d1 + d2 == 0 or d1 + d2 > 4:
                continue
            new_cols[f"x9^{d1}_x11^{d2}"] = (df["x9"] ** d1) * (df["x11"] ** d2)
    
    # Squares of all features
    for c in feature_cols:
        new_cols[f"{c}_sq"] = df[c] ** 2
    
    # Pairwise products
    for i in range(15):
        for j in range(i+1, 15):
            ci, cj = f"x{i}", f"x{j}"
            new_cols[f"{ci}_{cj}"] = df[ci] * df[cj]
    
    # x9 and x11 crossed with everything
    for c in feature_cols:
        if c not in ["x9", "x11"]:
            new_cols[f"x9sq_{c}"] = df["x9"] ** 2 * df[c]
            new_cols[f"x11sq_{c}"] = df["x11"] ** 2 * df[c]
            new_cols[f"x9_x11_{c}"] = df["x9"] * df["x11"] * df[c]
    
    # Concat all at once
    new_df = pd.DataFrame(new_cols, index=df.index)
    return pd.concat([df, new_df], axis=1)

X_tgt = make_targeted_features(X_imp)
X_test_tgt = make_targeted_features(X_test_imp)
print(f"  Feature count: {X_tgt.shape[1]}")

best_tgt_score = -np.inf
best_tgt_test = None
best_tgt_oof = None

for alpha in [1, 10, 100, 1000, 10000]:
    def make_t_ridge(a=alpha):
        return Pipeline([
            ("scale", StandardScaler()),
            ("ridge", Ridge(alpha=a)),
        ])
    oof, test_pred, r2 = cv_with_log(make_t_ridge, X_tgt, y, X_test_tgt, f"alpha={alpha}")
    if r2 > best_tgt_score:
        best_tgt_score = r2
        best_tgt_test = test_pred
        best_tgt_oof = oof



Target stats (raw):
  std=1978, min=-41008, max=69628
Target stats (after signed-log):
  std=3.96, min=-10.62, max=11.15

=== Model 1: Poly3 + Ridge (log-transformed target) ===
  alpha=0.1: R² = -7373.0162
  alpha=1: R² = -7514.5946
  alpha=10: R² = -8861.6081
  alpha=100: R² = -49102.7743
  alpha=1000: R² = -313855100.9225

Best Poly3+Ridge: alpha=0.1, R²=-7373.0162

=== Model 2: Targeted features + Ridge ===
  Feature count: 188
  alpha=1: R² = -7.2288
  alpha=10: R² = -5.8195
  alpha=100: R² = -1.0131
  alpha=1000: R² = 0.1407
  alpha=10000: R² = 0.0007


In [8]:

# ============================================================
# MODEL 3: LightGBM with log target + early stopping
# ============================================================
print("\n=== Model 3: LightGBM (log-transformed target) ===")

try:
    import lightgbm as lgb
    
    def cv_lgb_logtarget(X, y_orig, X_te, seed=42):
        oof = np.zeros(len(X))
        test_pred = np.zeros(len(X_te))
        y_log_target = signed_log(y_orig)
        
        for tr, va in kf.split(X):
            X_tr = X.iloc[tr] if hasattr(X, 'iloc') else X[tr]
            X_va = X.iloc[va] if hasattr(X, 'iloc') else X[va]
            y_tr = y_log_target.iloc[tr] if hasattr(y_log_target, 'iloc') else y_log_target[tr]
            y_va = y_log_target.iloc[va] if hasattr(y_log_target, 'iloc') else y_log_target[va]
            
            model = lgb.LGBMRegressor(
                objective="regression",
                n_estimators=2000,
                learning_rate=0.03,
                num_leaves=63,
                max_depth=-1,
                min_child_samples=5,
                subsample=0.85,
                subsample_freq=1,
                colsample_bytree=0.8,
                reg_lambda=0.1,
                random_state=seed,
                n_jobs=-1,
                verbose=-1,
            )
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                callbacks=[
                    lgb.early_stopping(stopping_rounds=50, verbose=False),
                    lgb.log_evaluation(0),
                ],
            )
            # Transform back to original scale
            oof[va] = signed_log_inverse(model.predict(X_va))
            test_pred += signed_log_inverse(model.predict(X_te)) / kf.n_splits
        
        return oof, test_pred, r2_score(y_orig, oof)
    
    lgb_oofs = []
    lgb_tests = []
    for seed in [42, 123, 456]:
        oof_s, test_s, r2_s = cv_lgb_logtarget(X, y, X_test, seed=seed)
        print(f"  seed={seed}: R² = {r2_s:.4f}")
        lgb_oofs.append(oof_s)
        lgb_tests.append(test_s)
    
    oof_lgb_avg = np.mean(lgb_oofs, axis=0)
    test_lgb_avg = np.mean(lgb_tests, axis=0)
    r2_lgb_avg = r2_score(y, oof_lgb_avg)
    print(f"  LGB multi-seed avg: R² = {r2_lgb_avg:.4f}")
    have_lgb = True
except ImportError:
    print("  LightGBM not installed!")
    have_lgb = False
    r2_lgb_avg = -np.inf
    test_lgb_avg = None
    oof_lgb_avg = None

# ============================================================
# COMPARE & BLEND
# ============================================================
print("\n" + "=" * 50)
print("=== Summary ===")
print(f"Poly3+Ridge:        R² = {best_poly_score:.4f}")
print(f"Targeted+Ridge:     R² = {best_tgt_score:.4f}")
if have_lgb:
    print(f"LightGBM multiseed: R² = {r2_lgb_avg:.4f}")



=== Model 3: LightGBM (log-transformed target) ===
  seed=42: R² = 0.0076
  seed=123: R² = 0.0094
  seed=456: R² = 0.0079
  LGB multi-seed avg: R² = 0.0083

=== Summary ===
Poly3+Ridge:        R² = -7373.0162
Targeted+Ridge:     R² = 0.1407
LightGBM multiseed: R² = 0.0083


In [9]:

candidates = [
    ("poly3", best_poly_test, best_poly_score, best_poly_oof),
    ("targeted", best_tgt_test, best_tgt_score, best_tgt_oof),
]
if have_lgb:
    candidates.append(("lgb", test_lgb_avg, r2_lgb_avg, oof_lgb_avg))

# Sort by score
candidates.sort(key=lambda x: -x[2])
print(f"\nBest single model: {candidates[0][0]} (R² = {candidates[0][2]:.4f})")

# Weighted blend (only models with positive R²)
positive = [c for c in candidates if c[2] > 0]
if len(positive) >= 2:
    weights = np.array([c[2] for c in positive])
    weights = weights / weights.sum()
    
    print(f"\nBlend weights:")
    for (name, _, score, _), w in zip(positive, weights):
        print(f"  {name}: weight={w:.3f} (R²={score:.4f})")
    
    blend_oof = np.zeros(len(y))
    blend_test = np.zeros(len(X_test))
    for (name, test_pred, _, oof), w in zip(positive, weights):
        blend_oof += w * oof
        blend_test += w * test_pred
    
    blend_r2 = r2_score(y, blend_oof)
    print(f"Blended OOF R²: {blend_r2:.4f}")
    
    if blend_r2 > candidates[0][2]:
        print(f"→ Using BLEND")
        final_predictions = blend_test
    else:
        print(f"→ Using best single: {candidates[0][0]}")
        final_predictions = candidates[0][1]
else:
    print(f"→ Using best single: {candidates[0][0]}")
    final_predictions = candidates[0][1]

# ============================================================
# Sanity check
# ============================================================
print("\n=== Sanity Check ===")
print(f"Train target: min={y.min():.1f}, max={y.max():.1f}, std={y.std():.1f}")
print(f"Predictions:  min={final_predictions.min():.1f}, max={final_predictions.max():.1f}, std={final_predictions.std():.1f}")
print(f"Std ratio (pred/target): {final_predictions.std()/y.std():.3f}")

# ============================================================
# Save submission to S3
# ============================================================
submission_pdf = pd.DataFrame({
    "Id": test_ids,
    "target": final_predictions
})

print(f"\nSubmission preview:")
print(submission_pdf.head())
print(f"Submission shape: {submission_pdf.shape}")

# Write to S3 via Spark (single CSV file)
schema = StructType([
    StructField("Id", IntegerType(), True),
    StructField("target", DoubleType(), True),
])
sub_spark = spark.createDataFrame(submission_pdf, schema=schema)

(sub_spark
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(OUTPUT_PATH))

print(f"\n✓ Submission written to: {OUTPUT_PATH}")

# Local backup
submission_pdf.to_csv("/tmp/submission.csv", index=False)
print(f"✓ Local copy: /tmp/submission.csv")


Best single model: targeted (R² = 0.1407)

Blend weights:
  targeted: weight=0.944 (R²=0.1407)
  lgb: weight=0.056 (R²=0.0083)
Blended OOF R²: 0.1360
→ Using best single: targeted

=== Sanity Check ===
Train target: min=-41008.0, max=69628.2, std=1978.2
Predictions:  min=-68112138.0, max=27931.7, std=1361976.7
Std ratio (pred/target): 688.501

Submission preview:
   Id     target
0   0 -15.489128
1   6 -15.700083
2   7  -8.563490
3   8   0.295808
4  12   1.530698
Submission shape: (2500, 2)

✓ Submission written to: s3://test-magangal/submission/
✓ Local copy: /tmp/submission.csv


In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql.types import DoubleType, IntegerType, StructType, StructField

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
TEST_PATH  = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_test.csv"
OUTPUT_PATH = "s3://test-magangal/submission/"

# Load
train_df = spark.read.csv(TRAIN_PATH, header=True, inferSchema=True).toPandas()
test_df  = spark.read.csv(TEST_PATH, header=True, inferSchema=True).toPandas()

feature_cols = [f"x{i}" for i in range(15)]
X = train_df[feature_cols].copy()
y = train_df["target"].copy()
X_test = test_df[feature_cols].copy()
test_ids = test_df["Id"].values

imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=feature_cols)

# ============================================================
# CRITICAL FIX: Cap log-space predictions before inverting
# ============================================================
# Find the max absolute value in train target's log space
y_log = np.sign(y) * np.log1p(np.abs(y))
LOG_CLIP = np.abs(y_log).max() * 1.1  # 10% buffer beyond observed
print(f"Log-space cap: ±{LOG_CLIP:.2f}")
print(f"Equivalent original-scale cap: ±{np.expm1(LOG_CLIP):.0f}")

def signed_log(arr):
    return np.sign(arr) * np.log1p(np.abs(arr))

def signed_log_inverse(arr):
    # Clip in log-space to prevent explosion
    arr_clipped = np.clip(arr, -LOG_CLIP, LOG_CLIP)
    return np.sign(arr_clipped) * np.expm1(np.abs(arr_clipped))

# ============================================================
# Build polynomial features manually (avoids huge memory)
# Focus on x9 and x11 since they dominate
# ============================================================
def build_features(df):
    """Build comprehensive feature set focused on x9 and x11."""
    new_cols = {}
    
    # All powers of single features up to degree 3
    for c in feature_cols:
        new_cols[f"{c}_sq"] = df[c] ** 2
        new_cols[f"{c}_cu"] = df[c] ** 3
    
    # All pairwise products
    for i in range(15):
        for j in range(i+1, 15):
            ci, cj = f"x{i}", f"x{j}"
            new_cols[f"{ci}_{cj}"] = df[ci] * df[cj]
    
    # Important triples (involving x9 or x11)
    for i in range(15):
        for j in range(i, 15):
            ci, cj = f"x{i}", f"x{j}"
            new_cols[f"x9_{ci}_{cj}"] = df["x9"] * df[ci] * df[cj]
            new_cols[f"x11_{ci}_{cj}"] = df["x11"] * df[ci] * df[cj]
    
    # x9-x11 cross terms  
    for c in feature_cols:
        new_cols[f"x9_x11_{c}"] = df["x9"] * df["x11"] * df[c]
    
    new_df = pd.DataFrame(new_cols, index=df.index)
    return pd.concat([df, new_df], axis=1)

print("Building features...")
X_full = build_features(X_imp)
X_test_full = build_features(X_test_imp)
print(f"Feature count: {X_full.shape[1]}")

# ============================================================
# CV setup
# ============================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_run(model_fn, X, y_orig, X_te, name, use_log=True):
    """Train model, optionally with log-transformed target."""
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_te))
    
    if use_log:
        y_train = signed_log(y_orig)
    else:
        y_train = y_orig
    
    for tr, va in kf.split(X):
        m = model_fn()
        Xtr = X.iloc[tr] if hasattr(X, 'iloc') else X[tr]
        Xva = X.iloc[va] if hasattr(X, 'iloc') else X[va]
        ytr = y_train.iloc[tr] if hasattr(y_train, 'iloc') else y_train[tr]
        m.fit(Xtr, ytr)
        
        if use_log:
            # Predict and invert with clipping
            pred_va_log = m.predict(Xva)
            pred_te_log = m.predict(X_te)
            oof[va] = signed_log_inverse(pred_va_log)
            test_pred += signed_log_inverse(pred_te_log) / kf.n_splits
        else:
            oof[va] = m.predict(Xva)
            test_pred += m.predict(X_te) / kf.n_splits
    
    score = r2_score(y_orig, oof)
    print(f"  {name}: R² = {score:.4f}")
    return oof, test_pred, score

# ============================================================
# MODEL 1: Ridge on raw target (no log) with engineered features
# Sometimes raw is better when outliers are real signal
# ============================================================
print("\n=== Model 1: Ridge on RAW target ===")
best_raw_score = -np.inf
best_raw_test = None
best_raw_oof = None

for alpha in [0.01, 0.1, 1, 10, 100, 1000, 10000]:
    def make_model(a=alpha):
        return Pipeline([
            ("scale", StandardScaler()),
            ("ridge", Ridge(alpha=a)),
        ])
    oof, test_pred, r2 = cv_run(make_model, X_full, y, X_test_full, f"alpha={alpha}", use_log=False)
    if r2 > best_raw_score:
        best_raw_score = r2
        best_raw_test = test_pred
        best_raw_oof = oof

# ============================================================
# MODEL 2: Ridge on LOG-transformed target (with clipping)
# ============================================================
print("\n=== Model 2: Ridge on LOG target (clipped) ===")
best_log_score = -np.inf
best_log_test = None
best_log_oof = None

for alpha in [0.01, 0.1, 1, 10, 100, 1000]:
    def make_model(a=alpha):
        return Pipeline([
            ("scale", StandardScaler()),
            ("ridge", Ridge(alpha=a)),
        ])
    oof, test_pred, r2 = cv_run(make_model, X_full, y, X_test_full, f"alpha={alpha}", use_log=True)
    if r2 > best_log_score:
        best_log_score = r2
        best_log_test = test_pred
        best_log_oof = oof

# ============================================================
# MODEL 3: LightGBM on raw target (with HUGE n_estimators carefully tuned)
# ============================================================
print("\n=== Model 3: LightGBM on RAW target ===")
try:
    import lightgbm as lgb
    
    def cv_lgb(X, y_target, X_te, seed=42):
        oof = np.zeros(len(X))
        test_pred = np.zeros(len(X_te))
        
        for tr, va in kf.split(X):
            X_tr = X.iloc[tr] if hasattr(X, 'iloc') else X[tr]
            X_va = X.iloc[va] if hasattr(X, 'iloc') else X[va]
            y_tr = y_target.iloc[tr] if hasattr(y_target, 'iloc') else y_target[tr]
            y_va = y_target.iloc[va] if hasattr(y_target, 'iloc') else y_target[va]
            
            model = lgb.LGBMRegressor(
                objective="regression",
                n_estimators=2000,
                learning_rate=0.05,
                num_leaves=31,            # less complex, prevents overfitting outliers
                max_depth=-1,
                min_child_samples=10,
                subsample=0.9,
                subsample_freq=1,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                random_state=seed,
                n_jobs=-1,
                verbose=-1,
            )
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                callbacks=[
                    lgb.early_stopping(stopping_rounds=50, verbose=False),
                    lgb.log_evaluation(0),
                ],
            )
            oof[va] = model.predict(X_va)
            test_pred += model.predict(X_te) / kf.n_splits
        
        return oof, test_pred
    
    # Run on RAW target (no log transform!)
    print("On raw target:")
    raw_lgb_oofs = []
    raw_lgb_tests = []
    for seed in [42, 123, 456]:
        oof_s, test_s = cv_lgb(X, y, X_test, seed=seed)
        r2_s = r2_score(y, oof_s)
        print(f"  seed={seed}: R² = {r2_s:.4f}")
        raw_lgb_oofs.append(oof_s)
        raw_lgb_tests.append(test_s)
    
    oof_lgb_raw = np.mean(raw_lgb_oofs, axis=0)
    test_lgb_raw = np.mean(raw_lgb_tests, axis=0)
    r2_lgb_raw = r2_score(y, oof_lgb_raw)
    print(f"  LGB raw avg: R² = {r2_lgb_raw:.4f}")
    
    # Also run on log target with clipping
    print("On log-transformed target (clipped):")
    y_log_full = signed_log(y)
    log_lgb_oofs = []
    log_lgb_tests = []
    for seed in [42, 123, 456]:
        oof_s_log, test_s_log = cv_lgb(X, y_log_full, X_test, seed=seed)
        # Invert with clipping
        oof_s = signed_log_inverse(oof_s_log)
        test_s = signed_log_inverse(test_s_log)
        r2_s = r2_score(y, oof_s)
        print(f"  seed={seed}: R² = {r2_s:.4f}")
        log_lgb_oofs.append(oof_s)
        log_lgb_tests.append(test_s)
    
    oof_lgb_log = np.mean(log_lgb_oofs, axis=0)
    test_lgb_log = np.mean(log_lgb_tests, axis=0)
    r2_lgb_log = r2_score(y, oof_lgb_log)
    print(f"  LGB log avg: R² = {r2_lgb_log:.4f}")
    
    have_lgb = True
except ImportError:
    have_lgb = False
    r2_lgb_raw = -np.inf
    r2_lgb_log = -np.inf

# ============================================================
# Compare and pick best
# ============================================================
print("\n" + "=" * 50)
print("=== Summary ===")
print(f"Ridge raw target:    R² = {best_raw_score:.4f}")
print(f"Ridge log target:    R² = {best_log_score:.4f}")
if have_lgb:
    print(f"LGB raw target:      R² = {r2_lgb_raw:.4f}")
    print(f"LGB log target:      R² = {r2_lgb_log:.4f}")

# Build candidates with sanity check on prediction range
all_models = []
all_models.append(("ridge_raw", best_raw_test, best_raw_score, best_raw_oof))
all_models.append(("ridge_log", best_log_test, best_log_score, best_log_oof))
if have_lgb:
    all_models.append(("lgb_raw", test_lgb_raw, r2_lgb_raw, oof_lgb_raw))
    all_models.append(("lgb_log", test_lgb_log, r2_lgb_log, oof_lgb_log))

# Sort by score
all_models.sort(key=lambda x: -x[2])

print(f"\nBest single: {all_models[0][0]} (R² = {all_models[0][2]:.4f})")
print(f"  Pred range: [{all_models[0][1].min():.1f}, {all_models[0][1].max():.1f}]")

# Try blending top models that have positive R²
positive = [m for m in all_models if m[2] > 0]
if len(positive) >= 2:
    weights = np.array([m[2] for m in positive])
    weights = weights / weights.sum()
    
    blend_oof = np.zeros(len(y))
    blend_test = np.zeros(len(X_test))
    for (name, test_pred, _, oof), w in zip(positive, weights):
        blend_oof += w * oof
        blend_test += w * test_pred
    
    blend_r2 = r2_score(y, blend_oof)
    print(f"\nWeighted blend R²: {blend_r2:.4f}")
    print(f"  Pred range: [{blend_test.min():.1f}, {blend_test.max():.1f}]")
    
    if blend_r2 > all_models[0][2]:
        final_predictions = blend_test
        print(f"→ Using BLEND")
    else:
        final_predictions = all_models[0][1]
        print(f"→ Using best single: {all_models[0][0]}")
else:
    final_predictions = all_models[0][1]

# Final sanity check
print("\n=== Sanity Check ===")
print(f"Train target: min={y.min():.1f}, max={y.max():.1f}, std={y.std():.1f}")
print(f"Predictions:  min={final_predictions.min():.1f}, max={final_predictions.max():.1f}, std={final_predictions.std():.1f}")

# Hard cap predictions to training target range (safety net)
y_min, y_max = y.min(), y.max()
final_predictions = np.clip(final_predictions, y_min * 1.5, y_max * 1.5)
print(f"After clip:   min={final_predictions.min():.1f}, max={final_predictions.max():.1f}")

# ============================================================
# Save
# ============================================================
submission_pdf = pd.DataFrame({"Id": test_ids, "target": final_predictions})
print(f"\nSubmission shape: {submission_pdf.shape}")
print(submission_pdf.head())

schema = StructType([
    StructField("Id", IntegerType(), True),
    StructField("target", DoubleType(), True),
])
sub_spark = spark.createDataFrame(submission_pdf, schema=schema)

(sub_spark
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(OUTPUT_PATH))

print(f"\n✓ Submission written to: {OUTPUT_PATH}")
submission_pdf.to_csv("/tmp/submission.csv", index=False)

Log-space cap: ±12.27
Equivalent original-scale cap: ±212358
Building features...
Feature count: 401

=== Model 1: Ridge on RAW target ===
  alpha=0.01: R² = -0.1913
  alpha=0.1: R² = -0.1912
  alpha=1: R² = -0.1905
  alpha=10: R² = -0.1837
  alpha=100: R² = -0.1339
  alpha=1000: R² = 0.0077
  alpha=10000: R² = 0.0701

=== Model 2: Ridge on LOG target (clipped) ===
  alpha=0.01: R² = -1.6211
  alpha=0.1: R² = -1.6214
  alpha=1: R² = -1.6247
  alpha=10: R² = -1.6772
  alpha=100: R² = -3.5459
  alpha=1000: R² = -3.3303

=== Model 3: LightGBM on RAW target ===
On raw target:
  seed=42: R² = 0.0654
  seed=123: R² = 0.0620
  seed=456: R² = 0.0647
  LGB raw avg: R² = 0.0656
On log-transformed target (clipped):
  seed=42: R² = 0.0088
  seed=123: R² = 0.0082
  seed=456: R² = 0.0074
  LGB log avg: R² = 0.0082

=== Summary ===
Ridge raw target:    R² = 0.0701
Ridge log target:    R² = -1.6211
LGB raw target:      R² = 0.0656
LGB log target:      R² = 0.0082

Best single: ridge_raw (R² = 0.0701)


In [12]:
import pandas as pd
import numpy as np

TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
TEST_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("=== SHAPES ===")
print(f"Train: {train.shape}")
print(f"Test: {test.shape}")

print("\n=== TRAIN COLUMNS ===")
print(train.columns.tolist())

print("\n=== TRAIN HEAD ===")
print(train.head())

print("\n=== TRAIN DESCRIBE ===")
print(train.describe())

print("\n=== MISSING VALUES (TRAIN) ===")
print(train.isna().sum())

print("\n=== MISSING VALUES (TEST) ===")
print(test.isna().sum())

print("\n=== TARGET DISTRIBUTION ===")
print(f"Min: {train['target'].min()}")
print(f"Max: {train['target'].max()}")
print(f"Mean: {train['target'].mean()}")
print(f"Median: {train['target'].median()}")
print(f"Std: {train['target'].std()}")

print("\n=== TARGET PERCENTILES ===")
for p in [0.1, 1, 5, 25, 50, 75, 95, 99, 99.9]:
    print(f"{p}%: {np.percentile(train['target'], p):.2f}")

print("\n=== TARGET EXTREME VALUES ===")
print("Top 10 largest |target|:")
print(train.nlargest(10, 'target')[['target']])
print("\nTop 10 smallest target:")
print(train.nsmallest(10, 'target')[['target']])

=== SHAPES ===
Train: (2500, 17)
Test: (2500, 16)

=== TRAIN COLUMNS ===
['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'target', 'Id']

=== TRAIN HEAD ===
          x0         x1        x2        x3  ...       x13       x14     target  Id
0        NaN   4.653737 -7.592557  1.714188  ...  4.670148  1.693087  10.147519   1
1  16.366673  -5.129070 -4.056004 -1.587978  ...  2.626949  0.458565  40.589306   2
2   1.671239  10.997672  1.770459  1.324817  ... -9.797043  3.805246 -29.214668   3
3 -12.860650  -4.226342  1.288700  1.143941  ...  5.625468  0.559720 -21.157257   4
4   5.276754  -5.392047  1.255338  1.220763  ...  5.439740  1.754827 -23.800982   5

[5 rows x 17 columns]

=== TRAIN DESCRIBE ===
                x0           x1  ...        target           Id
count  2468.000000  2470.000000  ...   2500.000000  2500.000000
mean      0.056332     0.121786  ...    -22.043467  2529.588000
std       7.845180    12.492723  ...   1978.177302  

In [14]:
import pandas as pd
import numpy as np
from itertools import combinations

TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
train = pd.read_csv(TRAIN_PATH)

features = [f'x{i}' for i in range(15)]
y = train['target'].values

# Fill NaN with median for analysis
train_filled = train[features].fillna(train[features].median())

print("=== LINEAR CORRELATIONS WITH TARGET ===")
for f in features:
    corr = np.corrcoef(train_filled[f], y)[0, 1]
    print(f"{f}: {corr:.4f}")

print("\n=== CORRELATIONS OF x_i^2 AND x_i^3 WITH TARGET ===")
for f in features:
    sq_corr = np.corrcoef(train_filled[f]**2, y)[0, 1]
    cube_corr = np.corrcoef(train_filled[f]**3, y)[0, 1]
    print(f"{f}: sq={sq_corr:.4f}, cube={cube_corr:.4f}")

print("\n=== TOP PAIRWISE PRODUCT CORRELATIONS ===")
pair_corrs = []
for i, j in combinations(range(15), 2):
    prod = train_filled[f'x{i}'] * train_filled[f'x{j}']
    corr = np.corrcoef(prod, y)[0, 1]
    pair_corrs.append((f'x{i}*x{j}', corr))

pair_corrs.sort(key=lambda x: abs(x[1]), reverse=True)
print("Top 20 pairwise products:")
for name, corr in pair_corrs[:20]:
    print(f"{name}: {corr:.4f}")

print("\n=== TOP TRIPLE PRODUCT CORRELATIONS ===")
triple_corrs = []
for i, j, k in combinations(range(15), 3):
    prod = train_filled[f'x{i}'] * train_filled[f'x{j}'] * train_filled[f'x{k}']
    corr = np.corrcoef(prod, y)[0, 1]
    triple_corrs.append((f'x{i}*x{j}*x{k}', corr))

triple_corrs.sort(key=lambda x: abs(x[1]), reverse=True)
print("Top 30 triple products:")
for name, corr in triple_corrs[:30]:
    print(f"{name}: {corr:.4f}")

print("\n=== CORRELATIONS WITH log|target| (sign preserved) ===")
y_signed_log = np.sign(y) * np.log1p(np.abs(y))
for f in features:
    corr = np.corrcoef(train_filled[f], y_signed_log)[0, 1]
    print(f"{f}: {corr:.4f}")

=== LINEAR CORRELATIONS WITH TARGET ===
x0: 0.0033
x1: 0.0049
x2: -0.0227
x3: -0.0020
x4: 0.0283
x5: -0.0262
x6: -0.0404
x7: 0.0204
x8: -0.0475
x9: 0.2319
x10: -0.0234
x11: 0.0274
x12: 0.0149
x13: 0.0029
x14: -0.0031

=== CORRELATIONS OF x_i^2 AND x_i^3 WITH TARGET ===
x0: sq=-0.0093, cube=0.0093
x1: sq=0.0220, cube=0.0134
x2: sq=0.0115, cube=-0.0170
x3: sq=-0.0260, cube=-0.0014
x4: sq=-0.0034, cube=0.0161
x5: sq=-0.0195, cube=-0.0270
x6: sq=-0.0151, cube=-0.0224
x7: sq=0.0191, cube=0.0103
x8: sq=0.0295, cube=-0.0383
x9: sq=0.0486, cube=0.4517
x10: sq=0.0133, cube=-0.0062
x11: sq=0.0753, cube=0.0802
x12: sq=0.0137, cube=0.0006
x13: sq=-0.0016, cube=0.0078
x14: sq=-0.0050, cube=-0.0045

=== TOP PAIRWISE PRODUCT CORRELATIONS ===
Top 20 pairwise products:
x9*x11: 0.1599
x9*x12: 0.1053
x2*x9: -0.1052
x0*x9: -0.0917
x8*x9: -0.0898
x5*x9: 0.0889
x1*x9: 0.0849
x7*x11: 0.0664
x1*x11: 0.0543
x1*x7: 0.0506
x9*x10: -0.0489
x3*x9: -0.0476
x11*x12: 0.0468
x1*x2: -0.0468
x3*x7: -0.0418
x0*x11: -0.04

In [15]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score

TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
train = pd.read_csv(TRAIN_PATH)

features = [f'x{i}' for i in range(15)]
y = train['target'].values
train_filled = train[features].fillna(train[features].median())

# Build comprehensive feature set: linear + squares + cubes + all pairs + all triples
print("Building feature matrix...")
feat_dict = {}

# Linear
for f in features:
    feat_dict[f] = train_filled[f].values

# Squares
for f in features:
    feat_dict[f'{f}^2'] = train_filled[f].values ** 2

# Cubes
for f in features:
    feat_dict[f'{f}^3'] = train_filled[f].values ** 3

# Pairs
for i, j in combinations(range(15), 2):
    feat_dict[f'x{i}*x{j}'] = train_filled[f'x{i}'].values * train_filled[f'x{j}'].values

# Triples
for i, j, k in combinations(range(15), 3):
    feat_dict[f'x{i}*x{j}*x{k}'] = (train_filled[f'x{i}'].values *
                                     train_filled[f'x{j}'].values *
                                     train_filled[f'x{k}'].values)

X_full = pd.DataFrame(feat_dict)
print(f"Total features: {X_full.shape[1]}")

# Fit ridge regression
print("\n=== RIDGE REGRESSION (alpha=1.0) ===")
ridge = Ridge(alpha=1.0)
ridge.fit(X_full.values, y)
pred = ridge.predict(X_full.values)
print(f"In-sample R²: {r2_score(y, pred):.4f}")

# Top coefficients by absolute value (normalized by feature std)
coef_importance = []
for col, coef in zip(X_full.columns, ridge.coef_):
    std = X_full[col].std()
    coef_importance.append((col, coef, coef * std))

coef_importance.sort(key=lambda x: abs(x[2]), reverse=True)
print("\nTop 40 features by coefficient*std (effective contribution):")
for name, coef, eff in coef_importance[:40]:
    print(f"{name}: coef={coef:.6f}, effective={eff:.4f}")

# Now try: maybe target = a*x9^3 + b*x4*x_something + ... 
# Test pure cubic of x9
print("\n=== SIMPLE FITS ===")
from sklearn.linear_model import LinearRegression

# x9^3 alone
lr = LinearRegression()
lr.fit(train_filled[['x9']].values**3, y)
pred = lr.predict(train_filled[['x9']].values**3)
print(f"target ~ x9^3: R² = {r2_score(y, pred):.4f}, coef={lr.coef_[0]:.4f}")

# x9^3 + x4 stuff
X_test_features = np.column_stack([
    train_filled['x9'].values**3,
    train_filled['x4'].values,
    train_filled['x4'].values**2,
    train_filled['x4'].values**3,
])
lr.fit(X_test_features, y)
pred = lr.predict(X_test_features)
print(f"target ~ x9^3 + x4 + x4^2 + x4^3: R² = {r2_score(y, pred):.4f}")

# Try target = product of 3 specific features
print("\n=== Testing if target = single triple product ===")
best_triple_r2 = -np.inf
best_triple_name = None
for i, j, k in combinations(range(15), 3):
    prod = train_filled[f'x{i}'].values * train_filled[f'x{j}'].values * train_filled[f'x{k}'].values
    lr.fit(prod.reshape(-1, 1), y)
    pred = lr.predict(prod.reshape(-1, 1))
    r2 = r2_score(y, pred)
    if r2 > best_triple_r2:
        best_triple_r2 = r2
        best_triple_name = f'x{i}*x{j}*x{k}'
        best_triple_coef = lr.coef_[0]

print(f"Best single triple: {best_triple_name}, R²={best_triple_r2:.4f}, coef={best_triple_coef:.4f}")

Building feature matrix...
Total features: 605

=== RIDGE REGRESSION (alpha=1.0) ===
In-sample R²: 0.6834

Top 40 features by coefficient*std (effective contribution):
x9^3: coef=0.633517, effective=922.4020
x2*x9*x11: coef=-0.361737, effective=-292.4792
x2*x8*x9: coef=0.666483, effective=258.2559
x7*x9*x11: coef=0.235900, effective=253.7534
x1*x3*x9: coef=-1.256297, effective=-226.3346
x9: coef=-32.369987, effective=-225.3225
x1*x4*x9: coef=-0.163337, effective=-188.9698
x1*x7*x9: coef=0.206244, effective=183.8312
x8*x9*x11: coef=-0.233080, effective=-171.2194
x5*x7*x9: coef=-0.148844, effective=-154.4838
x0*x9*x11: coef=-0.197845, effective=-153.4884
x6*x7*x9: coef=-0.380550, effective=-144.3043
x9*x10*x13: coef=-0.145405, effective=-141.6609
x7*x9*x14: coef=0.513425, effective=139.2299
x2*x9*x10: coef=0.185721, effective=138.5599
x3*x9*x11: coef=-0.677932, effective=-134.4146
x9*x12: coef=6.390493, effective=129.6369
x5*x9*x12: coef=0.431803, effective=128.2330
x1*x7*x12: coef=0.328

In [16]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.linear_model import Ridge, Lasso, LassoCV
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
train = pd.read_csv(TRAIN_PATH)

features = [f'x{i}' for i in range(15)]
y = train['target'].values
train_filled = train[features].fillna(train[features].median())

# Build feature matrix again
feat_dict = {}
for f in features:
    feat_dict[f] = train_filled[f].values
for f in features:
    feat_dict[f'{f}^2'] = train_filled[f].values ** 2
for f in features:
    feat_dict[f'{f}^3'] = train_filled[f].values ** 3
for i, j in combinations(range(15), 2):
    feat_dict[f'x{i}*x{j}'] = train_filled[f'x{i}'].values * train_filled[f'x{j}'].values
for i, j, k in combinations(range(15), 3):
    feat_dict[f'x{i}*x{j}*x{k}'] = (train_filled[f'x{i}'].values *
                                     train_filled[f'x{j}'].values *
                                     train_filled[f'x{k}'].values)

X_full = pd.DataFrame(feat_dict)
print(f"Total features: {X_full.shape[1]}")

# === Cross-validated Ridge to see true generalization ===
print("\n=== Ridge CV R² across alphas ===")
for alpha in [0.01, 0.1, 1.0, 10, 100, 1000, 10000]:
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr, va in kf.split(X_full):
        r = Ridge(alpha=alpha)
        r.fit(X_full.values[tr], y[tr])
        scores.append(r2_score(y[va], r.predict(X_full.values[va])))
    print(f"alpha={alpha}: CV R²={np.mean(scores):.4f} ± {np.std(scores):.4f}")

# === Lasso to find sparse formula ===
print("\n=== Lasso CV (find sparse formula) ===")
# Standardize features for Lasso
X_std = (X_full - X_full.mean()) / (X_full.std() + 1e-9)
lasso = LassoCV(cv=5, max_iter=10000, random_state=42, n_alphas=50)
lasso.fit(X_std.values, y)
pred = lasso.predict(X_std.values)
print(f"Lasso best alpha: {lasso.alpha_:.4f}")
print(f"Lasso in-sample R²: {r2_score(y, pred):.4f}")
print(f"Number of nonzero coefs: {np.sum(lasso.coef_ != 0)}")

# Show top Lasso features
nonzero_idx = np.where(lasso.coef_ != 0)[0]
lasso_feats = [(X_full.columns[i], lasso.coef_[i], lasso.coef_[i] * X_std.iloc[:, i].std()) 
               for i in nonzero_idx]
lasso_feats.sort(key=lambda x: abs(x[1]), reverse=True)
print(f"\nTop 50 Lasso features (after standardization):")
for name, coef, eff in lasso_feats[:50]:
    print(f"{name}: standardized_coef={coef:.4f}")

# === Try adding x_i^4 (quartic) features ===
print("\n=== Test if quartic x_i^4 helps ===")
X_quartic = X_full.copy()
for f in features:
    X_quartic[f'{f}^4'] = train_filled[f].values ** 4

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []
for tr, va in kf.split(X_quartic):
    r = Ridge(alpha=10)
    r.fit(X_quartic.values[tr], y[tr])
    scores.append(r2_score(y[va], r.predict(X_quartic.values[va])))
print(f"Ridge with quartics, CV R²={np.mean(scores):.4f}")

# === Test if target involves x_i * x_j^2 (mixed degree) ===
print("\n=== Adding x_i * x_j^2 features ===")
X_mixed = X_full.copy()
for i in range(15):
    for j in range(15):
        if i != j:
            X_mixed[f'x{i}*x{j}^2'] = train_filled[f'x{i}'].values * train_filled[f'x{j}'].values**2

print(f"Mixed features total: {X_mixed.shape[1]}")
scores = []
for tr, va in kf.split(X_mixed):
    r = Ridge(alpha=10)
    r.fit(X_mixed.values[tr], y[tr])
    scores.append(r2_score(y[va], r.predict(X_mixed.values[va])))
print(f"Ridge with x_i*x_j^2, CV R²={np.mean(scores):.4f}")

Total features: 605

=== Ridge CV R² across alphas ===
alpha=0.01: CV R²=-5.0959 ± 8.4291
alpha=0.1: CV R²=-5.0959 ± 8.4291
alpha=1.0: CV R²=-5.0959 ± 8.4291
alpha=10: CV R²=-5.0958 ± 8.4289
alpha=100: CV R²=-5.0947 ± 8.4275
alpha=1000: CV R²=-5.0846 ± 8.4132
alpha=10000: CV R²=-5.0152 ± 8.3093

=== Lasso CV (find sparse formula) ===
Lasso best alpha: 673.7150
Lasso in-sample R²: 0.0879
Number of nonzero coefs: 1

Top 50 Lasso features (after standardization):
x9^3: standardized_coef=219.5244

=== Test if quartic x_i^4 helps ===
Ridge with quartics, CV R²=-5.3056

=== Adding x_i * x_j^2 features ===
Mixed features total: 815
Ridge with x_i*x_j^2, CV R²=-6.7974
/home/hadoop/.local/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:1663: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an 

In [17]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.linear_model import HuberRegressor, RANSACRegressor, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
train = pd.read_csv(TRAIN_PATH)

features = [f'x{i}' for i in range(15)]
y = train['target'].values
train_filled = train[features].fillna(train[features].median())

# === Examine outliers: do extreme |target| correspond to extreme feature combinations? ===
print("=== TARGET OUTLIER ANALYSIS ===")
abs_y = np.abs(y)
threshold_99 = np.percentile(abs_y, 99)
threshold_95 = np.percentile(abs_y, 95)

print(f"99th percentile |target|: {threshold_99:.1f}")
print(f"95th percentile |target|: {threshold_95:.1f}")

# For extreme targets, what do the features look like?
extreme_mask = abs_y > threshold_99
normal_mask = abs_y < threshold_95

print(f"\nExtreme cases (|target|>{threshold_99:.0f}): {extreme_mask.sum()}")
print(f"Feature stats for extreme cases:")
print(train_filled[extreme_mask].abs().describe().loc[['mean', 'max']])

print(f"\nNormal cases stats:")
print(train_filled[normal_mask].abs().describe().loc[['mean']])

# === Check if max|x_i| correlates with extreme target ===
print("\n=== max(|x_i|) vs |target| ===")
max_abs = train_filled.abs().max(axis=1).values
print(f"Correlation of max|x_i| with |target|: {np.corrcoef(max_abs, abs_y)[0,1]:.4f}")
print(f"Correlation of max|x_i| with log|target|: {np.corrcoef(max_abs, np.log1p(abs_y))[0,1]:.4f}")

# Product of abs values of top 3 features per row
print("\n=== Top-3 |x_i| product ===")
sorted_abs = np.sort(train_filled.abs().values, axis=1)[:, -3:]  # top 3 per row
top3_prod = sorted_abs.prod(axis=1)
print(f"Correlation of top-3 |x_i| product with |target|: {np.corrcoef(top3_prod, abs_y)[0,1]:.4f}")
print(f"Correlation with log|target|: {np.corrcoef(top3_prod, np.log1p(abs_y))[0,1]:.4f}")

# === Build only the subset of features Ridge identified as important ===
print("\n=== TARGETED FEATURE SET (only important triples involving x9) ===")
# Based on previous Ridge: x9 is in almost every important interaction
important_feats = {}
# All features
for f in features:
    important_feats[f] = train_filled[f].values
    important_feats[f'{f}^2'] = train_filled[f].values ** 2
    important_feats[f'{f}^3'] = train_filled[f].values ** 3

# All pairs and triples involving x9
for j in range(15):
    if j != 9:
        important_feats[f'x9*x{j}'] = train_filled['x9'].values * train_filled[f'x{j}'].values
for j, k in combinations([i for i in range(15) if i != 9], 2):
    important_feats[f'x9*x{j}*x{k}'] = (train_filled['x9'].values * 
                                         train_filled[f'x{j}'].values * 
                                         train_filled[f'x{k}'].values)

X_targeted = pd.DataFrame(important_feats)
print(f"Targeted features: {X_targeted.shape[1]}")

# Cross-validated Ridge on this targeted set
print("\n=== Targeted Ridge CV ===")
for alpha in [1, 10, 100, 1000]:
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr, va in kf.split(X_targeted):
        r = Ridge(alpha=alpha)
        r.fit(X_targeted.values[tr], y[tr])
        scores.append(r2_score(y[va], r.predict(X_targeted.values[va])))
    print(f"alpha={alpha}: CV R²={np.mean(scores):.4f} ± {np.std(scores):.4f}")

# === Try Huber regression (robust to outliers) ===
print("\n=== Huber Regression CV (robust to outliers) ===")
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_targeted_std = scaler.fit_transform(X_targeted.values)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []
for tr, va in kf.split(X_targeted_std):
    h = HuberRegressor(max_iter=500, alpha=0.001)
    h.fit(X_targeted_std[tr], y[tr])
    scores.append(r2_score(y[va], h.predict(X_targeted_std[va])))
print(f"Huber CV R²={np.mean(scores):.4f} ± {np.std(scores):.4f}")

# === Try modeling on log|y| separately ===
print("\n=== Modeling log|target| with sign ===")
y_sign = np.sign(y)
y_log_abs = np.log1p(np.abs(y))

# Predict log|y|
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_log = []
scores_back = []
for tr, va in kf.split(X_targeted):
    r = Ridge(alpha=10)
    r.fit(X_targeted.values[tr], y_log_abs[tr])
    pred_log = r.predict(X_targeted.values[va])
    scores_log.append(r2_score(y_log_abs[va], pred_log))
    
    # Back-transform: need sign too
    r_sign = Ridge(alpha=10)
    r_sign.fit(X_targeted.values[tr], y[tr])
    pred_raw = r_sign.predict(X_targeted.values[va])
    pred_combined = np.sign(pred_raw) * (np.expm1(np.abs(pred_log)))
    scores_back.append(r2_score(y[va], pred_combined))

print(f"log|y| prediction R²: {np.mean(scores_log):.4f}")
print(f"Combined back-transform R²: {np.mean(scores_back):.4f}")

# === Where does the Ridge predict badly? ===
print("\n=== Residual analysis ===")
r = Ridge(alpha=10)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(y))
for tr, va in kf.split(X_targeted):
    r.fit(X_targeted.values[tr], y[tr])
    oof[va] = r.predict(X_targeted.values[va])

residuals = y - oof
print(f"Residual mean: {residuals.mean():.2f}")
print(f"Residual std: {residuals.std():.2f}")
print(f"Total var(y): {y.var():.2f}")
print(f"R² on full: {r2_score(y, oof):.4f}")

# R² excluding extreme outliers
mask_99 = abs_y < threshold_99
print(f"R² excluding top 1% |y|: {r2_score(y[mask_99], oof[mask_99]):.4f}")
mask_95 = abs_y < threshold_95
print(f"R² excluding top 5% |y|: {r2_score(y[mask_95], oof[mask_95]):.4f}")
mask_90 = abs_y < np.percentile(abs_y, 90)
print(f"R² excluding top 10% |y|: {r2_score(y[mask_90], oof[mask_90]):.4f}")

=== TARGET OUTLIER ANALYSIS ===
99th percentile |target|: 4759.7
95th percentile |target|: 307.8

Extreme cases (|target|>4760): 25
Feature stats for extreme cases:
             x0         x1        x2  ...       x12        x13       x14
mean   5.883834  12.073676   6.96366  ...  1.744824   8.610008  2.781626
max   16.910223  30.176348  14.32322  ...  6.060416  29.821986  9.123983

[2 rows x 15 columns]

Normal cases stats:
            x0        x1       x2  ...       x12      x13       x14
mean  6.125787  9.870639  6.19767  ...  2.352991  8.84736  3.036157

[1 rows x 15 columns]

=== max(|x_i|) vs |target| ===
Correlation of max|x_i| with |target|: 0.0426
Correlation of max|x_i| with log|target|: 0.1280

=== Top-3 |x_i| product ===
Correlation of top-3 |x_i| product with |target|: 0.0657
Correlation with log|target|: 0.1482

=== TARGETED FEATURE SET (only important triples involving x9) ===
Targeted features: 150

=== Targeted Ridge CV ===
alpha=1: CV R²=-3.0711 ± 5.3134
alpha=10: CV 

In [18]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
train = pd.read_csv(TRAIN_PATH)

features = [f'x{i}' for i in range(15)]
y = train['target'].values
train_filled = train[features].fillna(train[features].median())

# === STRATEGY: Train ONLY on non-outliers, see if model improves ===
abs_y = np.abs(y)
print("=== Train on bulk only, evaluate on bulk only ===")

# Create simple feature set first (linear + squares + cubes)
X_simple = pd.DataFrame()
for f in features:
    X_simple[f] = train_filled[f].values
    X_simple[f'{f}_sq'] = train_filled[f].values ** 2
    X_simple[f'{f}_cube'] = train_filled[f].values ** 3

# Add pairs and triples
for i, j in combinations(range(15), 2):
    X_simple[f'x{i}*x{j}'] = train_filled[f'x{i}'].values * train_filled[f'x{j}'].values
for i, j, k in combinations(range(15), 3):
    X_simple[f'x{i}*x{j}*x{k}'] = (train_filled[f'x{i}'].values * 
                                    train_filled[f'x{j}'].values * 
                                    train_filled[f'x{k}'].values)

print(f"Total features: {X_simple.shape[1]}")

# Filter to bulk (|y| < some threshold)
for thresh_pct in [90, 95, 99]:
    thresh = np.percentile(abs_y, thresh_pct)
    bulk_mask = abs_y < thresh
    
    print(f"\n--- Training on bottom {thresh_pct}% (|y|<{thresh:.0f}), n={bulk_mask.sum()} ---")
    
    # Within bulk, do CV
    bulk_idx = np.where(bulk_mask)[0]
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr_local, va_local in kf.split(bulk_idx):
        tr_idx = bulk_idx[tr_local]
        va_idx = bulk_idx[va_local]
        r = Ridge(alpha=100)
        r.fit(X_simple.values[tr_idx], y[tr_idx])
        pred = r.predict(X_simple.values[va_idx])
        scores.append(r2_score(y[va_idx], pred))
    print(f"Ridge CV R² on bulk-only: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

# === Try LightGBM with Huber loss ===
print("\n=== LightGBM with Huber objective ===")
import lightgbm as lgb

X_lgb = train[features].values  # Keep NaN for LGB

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(y))
for tr, va in kf.split(X_lgb):
    train_data = lgb.Dataset(X_lgb[tr], label=y[tr])
    val_data = lgb.Dataset(X_lgb[va], label=y[va], reference=train_data)
    model = lgb.train({
        'objective': 'huber',
        'alpha': 0.9,
        'learning_rate': 0.02,
        'num_leaves': 31,
        'min_child_samples': 20,
        'verbose': -1,
        'metric': 'rmse',
    }, train_data, num_boost_round=2000, valid_sets=[val_data],
       callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    oof[va] = model.predict(X_lgb[va])

print(f"LGB Huber R² (full): {r2_score(y, oof):.4f}")
mask_95 = abs_y < np.percentile(abs_y, 95)
print(f"LGB Huber R² (excl. top 5%): {r2_score(y[mask_95], oof[mask_95]):.4f}")

# === Try LightGBM with regression L1 (MAE) ===
print("\n=== LightGBM with L1 (MAE) objective ===")
oof_l1 = np.zeros(len(y))
for tr, va in kf.split(X_lgb):
    train_data = lgb.Dataset(X_lgb[tr], label=y[tr])
    val_data = lgb.Dataset(X_lgb[va], label=y[va], reference=train_data)
    model = lgb.train({
        'objective': 'regression_l1',
        'learning_rate': 0.02,
        'num_leaves': 31,
        'min_child_samples': 20,
        'verbose': -1,
        'metric': 'mae',
    }, train_data, num_boost_round=2000, valid_sets=[val_data],
       callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    oof_l1[va] = model.predict(X_lgb[va])

print(f"LGB L1 R² (full): {r2_score(y, oof_l1):.4f}")
print(f"LGB L1 R² (excl. top 5%): {r2_score(y[mask_95], oof_l1[mask_95]):.4f}")

# === Try LightGBM with default L2 ===
print("\n=== LightGBM with L2 objective (standard) ===")
oof_l2 = np.zeros(len(y))
for tr, va in kf.split(X_lgb):
    train_data = lgb.Dataset(X_lgb[tr], label=y[tr])
    val_data = lgb.Dataset(X_lgb[va], label=y[va], reference=train_data)
    model = lgb.train({
        'objective': 'regression',
        'learning_rate': 0.02,
        'num_leaves': 31,
        'min_child_samples': 20,
        'verbose': -1,
        'metric': 'rmse',
    }, train_data, num_boost_round=2000, valid_sets=[val_data],
       callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    oof_l2[va] = model.predict(X_lgb[va])

print(f"LGB L2 R² (full): {r2_score(y, oof_l2):.4f}")
print(f"LGB L2 R² (excl. top 5%): {r2_score(y[mask_95], oof_l2[mask_95]):.4f}")

# === Look at relationship between specific features and outliers ===
print("\n=== Profile of extreme outliers ===")
extreme_idx = np.argsort(np.abs(y))[-30:]
print("30 most extreme target rows:")
print(train.iloc[extreme_idx][['target'] + features].to_string())

# === Are extreme targets duplicated/clustered? ===
print("\n=== Target value distribution analysis ===")
print(f"Number of |y|>1000: {(abs_y > 1000).sum()}")
print(f"Number of |y|>5000: {(abs_y > 5000).sum()}")
print(f"Number of |y|>10000: {(abs_y > 10000).sum()}")

=== Train on bulk only, evaluate on bulk only ===
Total features: 605

--- Training on bottom 90% (|y|<123), n=2250 ---
Ridge CV R² on bulk-only: 0.9182 ± 0.0103

--- Training on bottom 95% (|y|<308), n=2375 ---
Ridge CV R² on bulk-only: 0.6913 ± 0.0374

--- Training on bottom 99% (|y|<4760), n=2475 ---
Ridge CV R² on bulk-only: -0.6986 ± 0.3403

=== LightGBM with Huber objective ===
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's rmse: 2256.65
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's rmse: 3466.14
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's rmse: 318.67
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's rmse: 980.799
Training until validation scores don't improve for 50 

In [25]:
import pandas as pd
import numpy as np
import boto3
from io import StringIO
from itertools import combinations
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
TEST_PATH  = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_test.csv"
BUCKET     = "test-magangal"
OUTPUT_KEY = "submission/submission.csv"

print("Loading data...")
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

features = ['x' + str(i) for i in range(15)]
y_train  = train['target'].values
test_ids = test['Id'].values

medians = train[features].median()
train_filled = train[features].fillna(medians)
test_filled  = test[features].fillna(medians)

def build_poly(df):
    feats = {}
    for f in features:
        feats[f]         = df[f].values
        feats[f + '_sq'] = df[f].values ** 2
        feats[f + '_cu'] = df[f].values ** 3
    for i, j in combinations(range(15), 2):
        name = 'x' + str(i) + '_x' + str(j)
        feats[name] = df['x' + str(i)].values * df['x' + str(j)].values
    for i, j, k in combinations(range(15), 3):
        name = 'x' + str(i) + '_x' + str(j) + '_x' + str(k)
        feats[name] = df['x' + str(i)].values * df['x' + str(j)].values * df['x' + str(k)].values
    return pd.DataFrame(feats)

print("Building polynomial features...")
X_poly      = build_poly(train_filled).values
X_test_poly = build_poly(test_filled).values
print("Features: " + str(X_poly.shape[1]))

abs_y = np.abs(y_train)

print("\n=== REALISTIC CV: Train on bulk, eval on FULL fold ===")
kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for pct in [80, 85, 88, 90, 92, 95]:
    thresh = np.percentile(abs_y, pct)
    bulk_mask = abs_y < thresh
    
    for alpha in [1, 10, 100, 1000]:
        oof = np.zeros(len(y_train))
        for tr, va in kf.split(X_poly):
            tr_bulk = tr[bulk_mask[tr]]
            r = Ridge(alpha=alpha)
            r.fit(X_poly[tr_bulk], y_train[tr_bulk])
            oof[va] = r.predict(X_poly[va])
        
        full_r2 = r2_score(y_train, oof)
        bulk_r2 = r2_score(y_train[bulk_mask], oof[bulk_mask])
        results[(pct, alpha)] = (full_r2, bulk_r2)
        print("  P" + str(pct) + " thresh=" + str(round(thresh,1)) +
              " alpha=" + str(alpha) +
              " FULL_R2=" + str(round(full_r2, 4)) +
              " BULK_R2=" + str(round(bulk_r2, 4)))

best_key = max(results, key=lambda k: results[k][0])
best_pct, best_alpha = best_key
best_full_r2, best_bulk_r2 = results[best_key]
best_thresh = np.percentile(abs_y, best_pct)

print("\n>>> BEST: P" + str(best_pct) + " thresh=" + str(round(best_thresh,1)) +
      " alpha=" + str(best_alpha) + " FULL_R2=" + str(round(best_full_r2,4)))

print("\n=== Generating final predictions ===")
bulk_mask = abs_y < best_thresh
oof = np.zeros(len(y_train))
test_pred = np.zeros(len(X_test_poly))
for tr, va in kf.split(X_poly):
    tr_bulk = tr[bulk_mask[tr]]
    r = Ridge(alpha=best_alpha)
    r.fit(X_poly[tr_bulk], y_train[tr_bulk])
    oof[va] = r.predict(X_poly[va])
    test_pred += r.predict(X_test_poly) / 5

print("OOF R2 (full): " + str(round(r2_score(y_train, oof), 4)))
print("OOF R2 (bulk): " + str(round(r2_score(y_train[bulk_mask], oof[bulk_mask]), 4)))

print("\n=== Comparison: Train on ALL data ===")
oof_all = np.zeros(len(y_train))
test_pred_all = np.zeros(len(X_test_poly))
for tr, va in kf.split(X_poly):
    r = Ridge(alpha=best_alpha)
    r.fit(X_poly[tr], y_train[tr])
    oof_all[va] = r.predict(X_poly[va])
    test_pred_all += r.predict(X_test_poly) / 5

print("OOF R2 (full, no filter): " + str(round(r2_score(y_train, oof_all), 4)))
print("OOF R2 (bulk, no filter): " + str(round(r2_score(y_train[bulk_mask], oof_all[bulk_mask]), 4)))

print("\n=== Blending bulk-trained + all-trained ===")
best_w = 0
best_blend_r2 = -np.inf
for w in np.arange(0, 1.01, 0.05):
    blend = w * oof + (1-w) * oof_all
    r2 = r2_score(y_train, blend)
    if r2 > best_blend_r2:
        best_blend_r2 = r2
        best_w = w
print("Best blend: bulk*" + str(round(best_w,2)) + " + all*" + str(round(1-best_w,2)) +
      " R2=" + str(round(best_blend_r2,4)))

candidates = {
    'bulk_only': (r2_score(y_train, oof), test_pred),
    'all_only':  (r2_score(y_train, oof_all), test_pred_all),
    'blend':     (best_blend_r2, best_w * test_pred + (1-best_w) * test_pred_all),
}
final_name = max(candidates, key=lambda k: candidates[k][0])
final_r2, final_test_pred = candidates[final_name]
print("\n>>> FINAL CHOICE: " + final_name + " with OOF R2=" + str(round(final_r2,4)))

submission = pd.DataFrame({'Id': test_ids, 'target': final_test_pred})
print("\nPrediction stats: min=" + str(round(final_test_pred.min())) +
      " max=" + str(round(final_test_pred.max())) +
      " mean=" + str(round(final_test_pred.mean(),2)))
print(submission.head())

csv_buffer = StringIO()
submission.to_csv(csv_buffer, index=False)
s3 = boto3.client('s3')
s3.put_object(Bucket=BUCKET, Key=OUTPUT_KEY, Body=csv_buffer.getvalue())
print("\nSaved to s3://" + BUCKET + "/" + OUTPUT_KEY)
submission.to_csv('submission.csv', index=False)

print("\n" + "="*60)
print("FINAL EXPECTED R2: " + str(round(final_r2, 4)))
print("="*60)

Loading data...
Building polynomial features...
Features: 605

=== REALISTIC CV: Train on bulk, eval on FULL fold ===
  P80 thresh=86.2 alpha=1 FULL_R2=0.0033 BULK_R2=0.914
  P80 thresh=86.2 alpha=10 FULL_R2=0.0033 BULK_R2=0.914
  P80 thresh=86.2 alpha=100 FULL_R2=0.0033 BULK_R2=0.9139
  P80 thresh=86.2 alpha=1000 FULL_R2=0.0032 BULK_R2=0.9104
  P85 thresh=100.1 alpha=1 FULL_R2=0.0032 BULK_R2=0.928
  P85 thresh=100.1 alpha=10 FULL_R2=0.0032 BULK_R2=0.928
  P85 thresh=100.1 alpha=100 FULL_R2=0.0031 BULK_R2=0.9279
  P85 thresh=100.1 alpha=1000 FULL_R2=0.003 BULK_R2=0.9255
  P88 thresh=111.8 alpha=1 FULL_R2=0.0028 BULK_R2=0.9176
  P88 thresh=111.8 alpha=10 FULL_R2=0.0028 BULK_R2=0.9177
  P88 thresh=111.8 alpha=100 FULL_R2=0.0028 BULK_R2=0.9177
  P88 thresh=111.8 alpha=1000 FULL_R2=0.0027 BULK_R2=0.9163
  P90 thresh=123.1 alpha=1 FULL_R2=0.0027 BULK_R2=0.9202
  P90 thresh=123.1 alpha=10 FULL_R2=0.0027 BULK_R2=0.9202
  P90 thresh=123.1 alpha=100 FULL_R2=0.0027 BULK_R2=0.9202
  P90 thresh=12

In [5]:
import sys
from io import StringIO

# AWS / data
import boto3
import numpy as np
import pandas as pd

# Spark
from pyspark.context import SparkContext
from pyspark.sql import SparkSession

# scikit-learn
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

In [6]:
from io import StringIO
# ------------------------------------------------------------
# Step 1: Paths
# ------------------------------------------------------------
TRAIN_PATH  = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
TEST_PATH   = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_test.csv"
OUTPUT_PATH = "s3://test-magangal/submission/"   # folder
OUTPUT_KEY  = "submission/submission.csv"         # final single file key
BUCKET      = "test-magangal"
# ------------------------------------------------------------
# Step 2: Load data with Spark, convert to pandas
# (dataset is small -> pandas is fine and lets us use sklearn)
# ------------------------------------------------------------
train_sdf = (spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .csv(TRAIN_PATH))

test_sdf  = (spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .csv(TEST_PATH))

train = train_sdf.toPandas()
test  = test_sdf.toPandas()

print("Train shape:", train.shape)
print("Test  shape:", test.shape)
print(train.columns.tolist())
# ------------------------------------------------------------
# Step 3: Prepare features
# ------------------------------------------------------------
feature_cols = [f"x{i}" for i in range(15)]

# Ensure numeric (Spark inferSchema sometimes leaves blanks/strings)
for c in feature_cols + ["target"]:
    if c in train.columns:
        train[c] = pd.to_numeric(train[c], errors="coerce")
for c in feature_cols:
    test[c] = pd.to_numeric(test[c], errors="coerce")

X       = train[feature_cols].values
y       = train["target"].values.astype(float)
X_test  = test[feature_cols].values
test_ids = test["Id"].values

# Look at target distribution (heavy-tailed outliers expected)
print("Target percentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  p{p:>2} = {np.nanpercentile(y, p):.2f}")

Train shape: (2500, 17)
Test  shape: (2500, 16)
['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'target', 'Id']
Target percentiles:
  p 1 = -2654.94
  p 5 = -127.63
  p25 = -43.17
  p50 = -0.77
  p75 = 42.00
  p95 = 120.91
  p99 = 1591.06
degree=1 alpha=   0.1: CV R2 = -0.4571
degree=1 alpha=     1: CV R2 = -0.4566
degree=1 alpha=     5: CV R2 = -0.4542
degree=1 alpha=    10: CV R2 = -0.4512
degree=1 alpha=    50: CV R2 = -0.4283
degree=1 alpha=   100: CV R2 = -0.4016
degree=1 alpha=   500: CV R2 = -0.2472
degree=1 alpha=  1000: CV R2 = -0.1410
degree=2 alpha=   0.1: CV R2 = -2.1223
degree=2 alpha=     1: CV R2 = -2.1201
degree=2 alpha=     5: CV R2 = -2.1102
degree=2 alpha=    10: CV R2 = -2.0979
degree=2 alpha=    50: CV R2 = -2.0038
degree=2 alpha=   100: CV R2 = -1.8952
degree=2 alpha=   500: CV R2 = -1.2789
degree=2 alpha=  1000: CV R2 = -0.8561

BEST -> degree=1, alpha=1000, CV R2=-0.1410
HistGB CV R2: -1.0268929758204064
Using Grad

In [ ]:

# ------------------------------------------------------------
# Step 4: Build model + tune alpha with cross-validation
# ------------------------------------------------------------
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

def make_model(alpha=10.0, degree=2):
    return Pipeline([
        ("imp",  SimpleImputer(strategy="median")),
        ("sc",   StandardScaler()),
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("sc2",  StandardScaler()),
        ("reg",  Ridge(alpha=alpha)),
    ])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for degree in [1, 2]:
    for alpha in [0.1, 1, 5, 10, 50, 100, 500, 1000]:
        scores = []
        for tr, va in kf.split(X):
            m = make_model(alpha=alpha, degree=degree)
            m.fit(X[tr], y[tr])
            scores.append(r2_score(y[va], m.predict(X[va])))
        results[(degree, alpha)] = np.mean(scores)
        print(f"degree={degree} alpha={alpha:>6}: CV R2 = {np.mean(scores):.4f}")

best = max(results, key=results.get)
print(f"\nBEST -> degree={best[0]}, alpha={best[1]}, CV R2={results[best]:.4f}")
# ------------------------------------------------------------
# Step 5: (Optional) compare with HistGradientBoosting
# ------------------------------------------------------------
from sklearn.ensemble import HistGradientBoostingRegressor

gb_scores = []
for tr, va in kf.split(X):
    gb = HistGradientBoostingRegressor(random_state=42, max_iter=400,
                                       learning_rate=0.05, max_depth=3)
    gb.fit(X[tr], y[tr])
    gb_scores.append(r2_score(y[va], gb.predict(X[va])))
print("HistGB CV R2:", np.mean(gb_scores))

# Pick whichever is better between best linear model and GB
use_gb = np.mean(gb_scores) > results[best]
print("Using GradientBoosting?" , use_gb)
# ------------------------------------------------------------
# Step 6: Train final model on ALL data and predict
# ------------------------------------------------------------
if use_gb:
    final = HistGradientBoostingRegressor(random_state=42, max_iter=400,
                                          learning_rate=0.05, max_depth=3)
    final.fit(X, y)
else:
    final = make_model(alpha=best[1], degree=best[0])
    final.fit(X, y)

preds = final.predict(X_test)
print("Predictions:", preds[:10])
print("Pred stats -> min:", preds.min(), "max:", preds.max(), "mean:", preds.mean())
# ------------------------------------------------------------
# Step 7: Build submission and write to S3 (single clean CSV)
# ------------------------------------------------------------
submission = pd.DataFrame({"Id": test_ids, "target": preds})
submission = submission.sort_values("Id").reset_index(drop=True)
print(submission.head())
print("Submission rows:", len(submission))

# Write directly to S3 as a single file using boto3 (avoids Spark part-files)
csv_buffer = StringIO()
submission.to_csv(csv_buffer, index=False)

s3 = boto3.client("s3")
s3.put_object(Bucket=BUCKET, Key=OUTPUT_KEY, Body=csv_buffer.getvalue())

print(f"Saved submission to s3://{BUCKET}/{OUTPUT_KEY}")

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

# ------------------------------------------------------------
# Robust CV helper
# ------------------------------------------------------------
def cv_eval(make_model, X, y, n=5, seed=42, transform_y=None, inverse_y=None):
    kf = KFold(n_splits=n, shuffle=True, random_state=seed)
    scores = []
    for tr, va in kf.split(X):
        m = make_model()
        ytr = y[tr]
        if transform_y is not None:
            ytr = transform_y(ytr)
        m.fit(X[tr], ytr)
        pred = m.predict(X[va])
        if inverse_y is not None:
            pred = inverse_y(pred)
        scores.append(r2_score(y[va], pred))
    return np.mean(scores), np.std(scores)

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)

# ============================================================
# Test 1: GradientBoosting with more trees / regularization
# ============================================================
def gb_strong():
    return HistGradientBoostingRegressor(
        random_state=42, max_iter=1000, learning_rate=0.03,
        max_depth=None, max_leaf_nodes=63, l2_regularization=1.0,
        min_samples_leaf=20)
print("GB strong:        ", cv_eval(gb_strong, X_imp, y))

# ============================================================
# Test 2: GB on log-transformed target (signed log handles tails)
# ============================================================
def signed_log(v):  return np.sign(v) * np.log1p(np.abs(v))
def signed_exp(v):  return np.sign(v) * (np.expm1(np.abs(v)))

print("GB + signed-log y:", cv_eval(gb_strong, X_imp, y,
                                    transform_y=signed_log, inverse_y=signed_exp))

# ============================================================
# Test 3: RandomForest
# ============================================================
def rf():
    return RandomForestRegressor(n_estimators=500, max_depth=None,
                                 min_samples_leaf=5, n_jobs=-1, random_state=42)
print("RandomForest:     ", cv_eval(rf, X_imp, y))
print("RF + signed-log y:", cv_eval(rf, X_imp, y,
                                    transform_y=signed_log, inverse_y=signed_exp))

GB strong:         (-0.8963061911621608, 1.6468102305794585)
GB + signed-log y: (0.0056267354311396066, 0.004952736156297621)
RandomForest:      (-1.6516752463660667, 2.9818452123763013)
RF + signed-log y: (0.0063320798592815695, 0.005768816935194722)


In [8]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)

def signed_log(v): return np.sign(v) * np.log1p(np.abs(v))
def signed_exp(v): return np.sign(v) * np.expm1(np.abs(v))

# ============================================================
# Diagnostic 1: How well do we predict in LOG space?
# (R2 on the transformed target - tells us if signal exists at all)
# ============================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
log_scores, orig_scores = [], []
for tr, va in kf.split(X_imp):
    m = HistGradientBoostingRegressor(random_state=42, max_iter=600,
                                      learning_rate=0.05, min_samples_leaf=15)
    ytr_log = signed_log(y[tr])
    m.fit(X_imp[tr], ytr_log)
    pred_log = m.predict(X_imp[va])
    yva_log = signed_log(y[va])
    log_scores.append(r2_score(yva_log, pred_log))           # R2 in log space
    orig_scores.append(r2_score(y[va], signed_exp(pred_log)))# R2 in orig space

print(f"R2 in LOG space:      {np.mean(log_scores):.4f}")
print(f"R2 in ORIGINAL space: {np.mean(orig_scores):.4f}")
# ============================================================
# Diagnostic 2: Correlation of each feature with target
# (in both raw and log space)
# ============================================================
import pandas as pd
yl = signed_log(y)
print("Feature correlations (raw y / log y):")
for i in range(15):
    xi = X_imp[:, i]
    c_raw = np.corrcoef(xi, y)[0,1]
    c_log = np.corrcoef(xi, yl)[0,1]
    print(f"  x{i:<2}: raw={c_raw:+.3f}   log={c_log:+.3f}")
# ============================================================
# Diagnostic 3: Does it predict the SIGN well?
# (maybe target sign is predictable even if magnitude isn't)
# ============================================================
from sklearn.metrics import accuracy_score
sign_acc = []
for tr, va in kf.split(X_imp):
    m = HistGradientBoostingRegressor(random_state=42, max_iter=400, min_samples_leaf=15)
    m.fit(X_imp[tr], signed_log(y[tr]))
    pred = m.predict(X_imp[va])
    sign_acc.append(accuracy_score(np.sign(y[va]), np.sign(pred)))
print(f"\nSign prediction accuracy: {np.mean(sign_acc):.3f}")

R2 in LOG space:      0.6398
R2 in ORIGINAL space: 0.0068
Feature correlations (raw y / log y):
  x0 : raw=+0.003   log=+0.047
  x1 : raw=+0.005   log=-0.130
  x2 : raw=-0.023   log=-0.183
  x3 : raw=-0.002   log=-0.052
  x4 : raw=+0.028   log=+0.611
  x5 : raw=-0.026   log=-0.307
  x6 : raw=-0.040   log=+0.108
  x7 : raw=+0.020   log=+0.074
  x8 : raw=-0.047   log=-0.232
  x9 : raw=+0.232   log=+0.205
  x10: raw=-0.023   log=+0.069
  x11: raw=+0.027   log=-0.021
  x12: raw=+0.015   log=-0.033
  x13: raw=+0.003   log=-0.014
  x14: raw=-0.003   log=+0.036

Sign prediction accuracy: 0.874


In [9]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)

def signed_log(v): return np.sign(v) * np.log1p(np.abs(v))
def signed_exp(v): return np.sign(v) * np.expm1(np.abs(v))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# APPROACH A: Predict in original space directly, but the model
# trains on log-target then we BLEND. First, baseline check:
# train directly on original y (no transform) - we know this fails
# ============================================================

# ============================================================
# APPROACH B: Two-stage - predict log-magnitude AND predict sign,
# then reconstruct. But key: clip extreme predictions.
# ============================================================
def reconstruct_clipped(pred_log, clip_q):
    pred = signed_exp(pred_log)
    # clip to training-data range to avoid explosion
    lo, hi = clip_q
    return np.clip(pred, lo, hi)

# ============================================================
# APPROACH C (BEST): Train on log-target, but predict original.
# Then find the optimal SHRINKAGE factor that maximizes R2 in
# original space (huge predictions get pulled toward mean).
# ============================================================
print("Testing shrinkage on reconstructed predictions:\n")

for shrink in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2]:
    scores = []
    for tr, va in kf.split(X_imp):
        m = HistGradientBoostingRegressor(random_state=42, max_iter=600,
                                          learning_rate=0.05, min_samples_leaf=15)
        m.fit(X_imp[tr], signed_log(y[tr]))
        pred = signed_exp(m.predict(X_imp[va])) * shrink
        scores.append(r2_score(y[va], pred))
    print(f"  shrink={shrink}: R2 = {np.mean(scores):.4f}")
# ============================================================
# APPROACH D: Clip predictions to a percentile range
# ============================================================
print("\nTesting clipping of reconstructed predictions:\n")
lo_all = np.percentile(y, 1)
hi_all = np.percentile(y, 99)

for clip_pct in [(0.5,99.5),(1,99),(2,98),(5,95),(10,90)]:
    scores = []
    for tr, va in kf.split(X_imp):
        m = HistGradientBoostingRegressor(random_state=42, max_iter=600,
                                          learning_rate=0.05, min_samples_leaf=15)
        m.fit(X_imp[tr], signed_log(y[tr]))
        pred = signed_exp(m.predict(X_imp[va]))
        lo = np.percentile(y[tr], clip_pct[0])
        hi = np.percentile(y[tr], clip_pct[1])
        pred = np.clip(pred, lo, hi)
        scores.append(r2_score(y[va], pred))
    print(f"  clip={clip_pct}: R2 = {np.mean(scores):.4f}")
# ============================================================
# APPROACH E: Optimal shrink + clip combined
# ============================================================
print("\nShrink + clip combined:\n")
best_r2, best_cfg = -np.inf, None
for shrink in [1.0, 0.8, 0.6, 0.5, 0.4]:
    for clip_pct in [(1,99),(2,98),(5,95)]:
        scores = []
        for tr, va in kf.split(X_imp):
            m = HistGradientBoostingRegressor(random_state=42, max_iter=600,
                                              learning_rate=0.05, min_samples_leaf=15)
            m.fit(X_imp[tr], signed_log(y[tr]))
            pred = signed_exp(m.predict(X_imp[va])) * shrink
            lo = np.percentile(y[tr], clip_pct[0])
            hi = np.percentile(y[tr], clip_pct[1])
            pred = np.clip(pred, lo, hi)
            r2 = r2_score(y[va], pred)
            scores.append(r2)
        mr2 = np.mean(scores)
        if mr2 > best_r2:
            best_r2, best_cfg = mr2, (shrink, clip_pct)
        print(f"  shrink={shrink}, clip={clip_pct}: R2 = {mr2:.4f}")

print(f"\nBEST: {best_cfg} -> R2 = {best_r2:.4f}")

Testing shrinkage on reconstructed predictions:

  shrink=1.0: R2 = 0.0068
  shrink=0.9: R2 = 0.0073
  shrink=0.8: R2 = 0.0075
  shrink=0.7: R2 = 0.0075
  shrink=0.6: R2 = 0.0071
  shrink=0.5: R2 = 0.0064
  shrink=0.4: R2 = 0.0053
  shrink=0.3: R2 = 0.0040
  shrink=0.2: R2 = 0.0024

Testing clipping of reconstructed predictions:

  clip=(0.5, 99.5): R2 = 0.0068
  clip=(1, 99): R2 = 0.0068
  clip=(2, 98): R2 = 0.0064
  clip=(5, 95): R2 = 0.0063
  clip=(10, 90): R2 = 0.0061

Shrink + clip combined:

  shrink=1.0, clip=(1, 99): R2 = 0.0068
  shrink=1.0, clip=(2, 98): R2 = 0.0064
  shrink=1.0, clip=(5, 95): R2 = 0.0063
  shrink=0.8, clip=(1, 99): R2 = 0.0075
  shrink=0.8, clip=(2, 98): R2 = 0.0075
  shrink=0.8, clip=(5, 95): R2 = 0.0065
  shrink=0.6, clip=(1, 99): R2 = 0.0071
  shrink=0.6, clip=(2, 98): R2 = 0.0071
  shrink=0.6, clip=(5, 95): R2 = 0.0062
  shrink=0.5, clip=(1, 99): R2 = 0.0064
  shrink=0.5, clip=(2, 98): R2 = 0.0064
  shrink=0.5, clip=(5, 95): R2 = 0.0058
  shrink=0.4, cli

In [11]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)

# ============================================================
# Diagnostic: how concentrated is the variance?
# ============================================================
y_centered = y - y.mean()
ss_total = np.sum(y_centered**2)
order = np.argsort(-np.abs(y_centered))
cum = np.cumsum(y_centered[order]**2) / ss_total
for k in [1, 2, 5, 10, 20, 50, 100]:
    print(f"  Top {k:>3} points explain {cum[k-1]*100:.1f}% of total variance")
print(f"\nMax |y|: {np.abs(y).max():.0f}")
print(f"Total points: {len(y)}")

  Top   1 points explain 49.6% of total variance
  Top   2 points explain 66.8% of total variance
  Top   5 points explain 79.6% of total variance
  Top  10 points explain 88.5% of total variance
  Top  20 points explain 94.4% of total variance
  Top  50 points explain 98.9% of total variance
  Top 100 points explain 99.8% of total variance

Max |y|: 69628
Total points: 2500


In [4]:
# ============================================================
# SELF-CONTAINED: reload data + run giant-prediction diagnostics
# ============================================================
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor

TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
TEST_PATH  = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_test.csv"

train = spark.read.option("header","true").option("inferSchema","true").csv(TRAIN_PATH).toPandas()
test  = spark.read.option("header","true").option("inferSchema","true").csv(TEST_PATH).toPandas()

feature_cols = [f"x{i}" for i in range(15)]
for c in feature_cols + ["target"]:
    if c in train.columns: train[c] = pd.to_numeric(train[c], errors="coerce")
for c in feature_cols:
    test[c] = pd.to_numeric(test[c], errors="coerce")

X        = train[feature_cols].values
y        = train["target"].values.astype(float)
X_test   = test[feature_cols].values
test_ids = test["Id"].values

print("Loaded:", X.shape, y.shape, X_test.shape)

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
X_test_imp = imp.transform(X_test)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def signed_log(v): return np.sign(v)*np.log1p(np.abs(v))
def signed_exp(v): return np.sign(v)*np.expm1(np.abs(v))
# ============================================================
# Biggest actual vs predicted (log-recon)
# ============================================================
all_true, all_pred = [], []
for tr, va in kf.split(X_imp):
    m = HistGradientBoostingRegressor(random_state=42, max_iter=800,
                                      learning_rate=0.05, min_samples_leaf=10)
    m.fit(X_imp[tr], signed_log(y[tr]))
    all_true.append(y[va]); all_pred.append(signed_exp(m.predict(X_imp[va])))
all_true=np.concatenate(all_true); all_pred=np.concatenate(all_pred)

big = np.argsort(-np.abs(all_true))[:15]
print("BIGGEST ACTUAL vs PREDICTED (log-recon):")
for i in big:
    print(f"  actual={all_true[i]:>12.1f}   pred={all_pred[i]:>12.1f}")

print(f"\nCorr(log|pred|,log|actual|): {np.corrcoef(np.log1p(np.abs(all_pred)),np.log1p(np.abs(all_true)))[0,1]:.3f}")
# ============================================================
# Power-scaling calibration
# ============================================================
print("=== Power-scaled reconstruction ===")
best=-np.inf; bestp=None
for power in [1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.5, 3.0]:
    scores=[]
    for tr,va in kf.split(X_imp):
        m=HistGradientBoostingRegressor(random_state=42,max_iter=800,
            learning_rate=0.05,min_samples_leaf=10)
        m.fit(X_imp[tr], signed_log(y[tr]))
        plog=m.predict(X_imp[va])
        pred=np.sign(plog)*np.expm1(np.abs(plog)*power)
        scores.append(r2_score(y[va],pred))
    s=np.mean(scores)
    if s>best: best,bestp=s,power
    print(f"  power={power}: R2={s:.4f}")
print(f"BEST power={bestp} R2={best:.4f}")

Loaded: (2500, 15) (2500,) (2500, 15)
BIGGEST ACTUAL vs PREDICTED (log-recon):
  actual=     69628.2   pred=       247.0
  actual=    -41008.0   pred=      -271.0
  actual=     24159.1   pred=        17.4
  actual=    -20132.1   pred=       103.6
  actual=    -16236.5   pred=         0.3
  actual=    -14910.1   pred=        -3.5
  actual=    -13690.3   pred=        -4.7
  actual=     13608.8   pred=       -95.4
  actual=    -12882.7   pred=      -143.3
  actual=    -10592.2   pred=         0.5
  actual=     -9534.4   pred=         3.8
  actual=      9529.8   pred=        51.3
  actual=     -8793.8   pred=         5.6
  actual=     -8400.0   pred=       -92.2
  actual=     -7595.8   pred=      -286.6

Corr(log|pred|,log|actual|): 0.497
=== Power-scaled reconstruction ===
  power=1.0: R2=0.0053
  power=1.2: R2=-0.0763
  power=1.4: R2=-1.1497
  power=1.6: R2=-13.2042
  power=1.8: R2=-149.3119
  power=2.0: R2=-1747.9703
  power=2.5: R2=-987361.3665
  power=3.0: R2=-687517256.1080
BEST powe

In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def signed_log(v): return np.sign(v)*np.log1p(np.abs(v))
def signed_exp(v): return np.sign(v)*np.expm1(np.abs(v))

# ============================================================
# LINEAR REGRESSION on log-target (in log space - where R2=0.64!)
# Recall: tree model got 0.64 in log space. Can LINEAR do well too?
# ============================================================
print("LINEAR models, evaluated in LOG space:")
for alpha in [0.01, 0.1, 1, 10, 100]:
    scores=[]
    for tr,va in kf.split(X_imp):
        sc = StandardScaler().fit(X_imp[tr])
        m = Ridge(alpha=alpha)
        m.fit(sc.transform(X_imp[tr]), signed_log(y[tr]))
        pred_log = m.predict(sc.transform(X_imp[va]))
        scores.append(r2_score(signed_log(y[va]), pred_log))
    print(f"  Ridge alpha={alpha}: LOG-space R2 = {np.mean(scores):.4f}")


LINEAR models, evaluated in LOG space:
Ridge(alpha=0.01)
Ridge(alpha=0.01)
Ridge(alpha=0.01)
Ridge(alpha=0.01)
Ridge(alpha=0.01)
  Ridge alpha=0.01: LOG-space R2 = 0.6338
Ridge(alpha=0.1)
Ridge(alpha=0.1)
Ridge(alpha=0.1)
Ridge(alpha=0.1)
Ridge(alpha=0.1)
  Ridge alpha=0.1: LOG-space R2 = 0.6338
Ridge(alpha=1)
Ridge(alpha=1)
Ridge(alpha=1)
Ridge(alpha=1)
Ridge(alpha=1)
  Ridge alpha=1: LOG-space R2 = 0.6338
Ridge(alpha=10)
Ridge(alpha=10)
Ridge(alpha=10)
Ridge(alpha=10)
Ridge(alpha=10)
  Ridge alpha=10: LOG-space R2 = 0.6338
Ridge(alpha=100)
Ridge(alpha=100)
Ridge(alpha=100)
Ridge(alpha=100)
Ridge(alpha=100)
  Ridge alpha=100: LOG-space R2 = 0.6325


In [6]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# STEP 1: which feature drives the GIANT magnitudes?
# Look at the rows with biggest |y| and inspect their features
# ============================================================
big_idx = np.argsort(-np.abs(y))[:15]
print("Features of the 15 biggest |y| rows:")
cols = [f"x{i}" for i in range(15)]
print("  y          " + "".join(f"{c:>9}" for c in cols))
for i in big_idx:
    print(f"  {y[i]:>9.0f}  " + "".join(f"{X_imp[i,j]:>9.2f}" for j in range(15)))
# ============================================================
# STEP 2: for each feature, check if 1/feature explains giants
# (ratios produce huge values when denominator -> 0)
# ============================================================
print("\nWhich features are NEAR ZERO for the giant rows?")
print("(a near-zero feature used as denominator creates giants)")
for i in big_idx[:8]:
    near_zero = [(j, X_imp[i,j]) for j in range(15) if abs(X_imp[i,j])<2.0]
    print(f"  y={y[i]:>9.0f}: near-zero feats -> {[(f'x{j}',round(v,2)) for j,v in near_zero]}")
# ============================================================
# STEP 3: test simple ratio / product hypotheses directly
# correlate candidate engineered features with y
# ============================================================
print("\nCorrelation of engineered features with raw y:")
eps = 1e-6
candidates = {}
for i in range(15):
    for j in range(15):
        if i==j: continue
        # ratio xi / xj
        feat = X_imp[:,i] / (X_imp[:,j] + np.sign(X_imp[:,j]+eps)*eps)
        feat = np.clip(feat, -1e6, 1e6)
        c = np.corrcoef(feat, y)[0,1]
        if np.abs(c) > 0.2:
            candidates[f"x{i}/x{j}"] = c

for k,v in sorted(candidates.items(), key=lambda x:-abs(x[1]))[:20]:
    print(f"  {k}: corr={v:+.3f}")


Features of the 15 biggest |y| rows:
  y                 x0       x1       x2       x3       x4       x5       x6       x7       x8       x9      x10      x11      x12      x13      x14
      69628      -7.30    17.74   -11.47    -1.30    -0.47     2.23    -3.22    15.33    -9.56    28.52   -14.70    42.74     3.24    -1.25     0.89
     -41008      -7.04    11.51    10.01    -2.15    -2.31    28.25     5.78   -16.93     8.14   -23.89     8.53   -12.39     1.37     2.79     0.53
      24159      -4.25   -15.92   -11.55     1.14     4.84    15.84    -1.44   -15.01   -16.41    20.02     7.32     1.98     0.83   -16.20    -3.12
     -20132       4.77   -18.73    -5.97     4.68    24.35    18.05     0.23     7.89    -2.68   -18.88    -7.92    -6.85     0.08     5.46     6.49
     -16237       4.57    -2.41    -5.28     2.12    -2.02   -18.55     5.36    -7.62     6.06   -17.55    -5.12    16.77     0.02     6.77    -3.54
     -14910      -0.48     6.20   -13.95    -2.17    -6.07    19.79  

In [7]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)

# ============================================================
# CONFIRM: x9 vs log|y|
# ============================================================
logy = np.log(np.abs(y)+1e-9)
print("Correlation of each feature with log|y|:")
for i in range(15):
    c = np.corrcoef(X_imp[:,i], logy)[0,1]
    print(f"  x{i:<2}: corr with log|y| = {c:+.3f}")

print(f"\nCorr(x9, log|y|): {np.corrcoef(X_imp[:,9], logy)[0,1]:+.3f}")
print(f"Corr(|x9|, log|y|): {np.corrcoef(np.abs(X_imp[:,9]), logy)[0,1]:+.3f}")
print(f"Sign agreement(x9, y): {np.mean(np.sign(X_imp[:,9])==np.sign(y)):.3f}")
# ============================================================
# Fit log|y| as LINEAR function of features -> R2 in log space
# ============================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores=[]
for tr,va in kf.split(X_imp):
    m = LinearRegression().fit(X_imp[tr], np.log(np.abs(y[tr])+1e-9))
    pred = m.predict(X_imp[va])
    scores.append(r2_score(np.log(np.abs(y[va])+1e-9), pred))
print(f"LinearRegression predicting log|y|: R2 = {np.mean(scores):.4f}")

# show coefficients - which features matter
m = LinearRegression().fit(X_imp, np.log(np.abs(y)+1e-9))
print("\nCoefficients for log|y|:")
for i in range(15):
    print(f"  x{i:<2}: {m.coef_[i]:+.4f}")
print(f"  intercept: {m.intercept_:+.4f}")
# ============================================================
# THE FULL MODEL: predict log|y| linearly, predict sign,
# reconstruct as sign * exp(log_mag), evaluate in ORIGINAL space
# ============================================================
from sklearn.linear_model import LogisticRegression

scores=[]
for tr,va in kf.split(X_imp):
    # magnitude model (linear on log|y|)
    mag = LinearRegression().fit(X_imp[tr], np.log(np.abs(y[tr])+1e-9))
    log_mag = mag.predict(X_imp[va])
    # sign model
    sgn_clf = LogisticRegression(max_iter=1000).fit(X_imp[tr], (y[tr]>0).astype(int))
    sgn = np.where(sgn_clf.predict(X_imp[va])==1, 1, -1)
    # reconstruct
    pred = sgn * np.exp(log_mag)
    scores.append(r2_score(y[va], pred))
print(f"FULL reconstruction R2 (original space): {np.mean(scores):.4f}")

Correlation of each feature with log|y|:
  x0 : corr with log|y| = -0.006
  x1 : corr with log|y| = -0.003
  x2 : corr with log|y| = -0.021
  x3 : corr with log|y| = -0.012
  x4 : corr with log|y| = +0.024
  x5 : corr with log|y| = -0.007
  x6 : corr with log|y| = -0.008
  x7 : corr with log|y| = -0.028
  x8 : corr with log|y| = -0.015
  x9 : corr with log|y| = -0.028
  x10: corr with log|y| = -0.033
  x11: corr with log|y| = -0.005
  x12: corr with log|y| = +0.015
  x13: corr with log|y| = +0.005
  x14: corr with log|y| = -0.027

Corr(x9, log|y|): -0.028
Corr(|x9|, log|y|): +0.145
Sign agreement(x9, y): 0.558
LinearRegression predicting log|y|: R2 = -0.0127

Coefficients for log|y|:
  x0 : -0.0010
  x1 : -0.0002
  x2 : -0.0033
  x3 : -0.0084
  x4 : +0.0024
  x5 : -0.0007
  x6 : -0.0026
  x7 : -0.0038
  x8 : -0.0032
  x9 : -0.0055
  x10: -0.0033
  x11: -0.0006
  x12: +0.0078
  x13: +0.0004
  x14: -0.0099
  intercept: +3.6653
FULL reconstruction R2 (original space): 0.0042


In [8]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)

# ============================================================
# Fit linear regression on data with outlier rows REMOVED.
# Test different thresholds for "what counts as a corrupt giant"
# Evaluate R2 on the CLEAN validation points (within threshold)
# ============================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Removing |y| > threshold, fit LinearRegression:")
for thresh in [50, 75, 100, 150, 200, 300, 500, 1000]:
    scores = []
    for tr, va in kf.split(X_imp):
        # clean training set
        mask_tr = np.abs(y[tr]) <= thresh
        m = LinearRegression().fit(X_imp[tr][mask_tr], y[tr][mask_tr])
        # evaluate on clean validation points
        mask_va = np.abs(y[va]) <= thresh
        pred = m.predict(X_imp[va][mask_va])
        scores.append(r2_score(y[va][mask_va], pred))
    print(f"  thresh=±{thresh}: clean R2 = {np.mean(scores):.4f}  (kept {np.mean(np.abs(y)<=thresh)*100:.1f}% of data)")
# ============================================================
# Same, but evaluate on ALL validation points (full test set
# will have giants too - this is what the leaderboard sees)
# ============================================================
print("\nFit on clean train, evaluate on FULL validation:")
for thresh in [50, 75, 100, 150, 200, 300, 500, 1000]:
    scores = []
    for tr, va in kf.split(X_imp):
        mask_tr = np.abs(y[tr]) <= thresh
        m = LinearRegression().fit(X_imp[tr][mask_tr], y[tr][mask_tr])
        pred = m.predict(X_imp[va])      # predict ALL validation
        scores.append(r2_score(y[va], pred))   # score against ALL (incl giants)
    print(f"  thresh=±{thresh}: full R2 = {np.mean(scores):.4f}")
# ============================================================
# Check the linear fit quality on clean data - look at coefs
# ============================================================
mask = np.abs(y) <= 200
m = LinearRegression().fit(X_imp[mask], y[mask])
print(f"\nLinear fit on clean data (|y|<=200, {mask.sum()} rows):")
print(f"Train R2: {m.score(X_imp[mask], y[mask]):.4f}")
print("\nCoefficients:")
for i in range(15):
    print(f"  x{i:<2}: {m.coef_[i]:+8.3f}")
print(f"  intercept: {m.intercept_:+.3f}")

Removing |y| > threshold, fit LinearRegression:
  thresh=±50: clean R2 = 0.9180  (kept 57.2% of data)
  thresh=±75: clean R2 = 0.9414  (kept 75.4% of data)
  thresh=±100: clean R2 = 0.9568  (kept 84.9% of data)
  thresh=±150: clean R2 = 0.9399  (kept 92.7% of data)
  thresh=±200: clean R2 = 0.8958  (kept 94.2% of data)
  thresh=±300: clean R2 = 0.8217  (kept 94.9% of data)
  thresh=±500: clean R2 = 0.6896  (kept 95.5% of data)
  thresh=±1000: clean R2 = 0.3245  (kept 96.6% of data)

Fit on clean train, evaluate on FULL validation:
  thresh=±50: full R2 = 0.0047
  thresh=±75: full R2 = 0.0048
  thresh=±100: full R2 = 0.0048
  thresh=±150: full R2 = 0.0050
  thresh=±200: full R2 = 0.0051
  thresh=±300: full R2 = 0.0055
  thresh=±500: full R2 = 0.0066
  thresh=±1000: full R2 = 0.0093

Linear fit on clean data (|y|<=200, 2354 rows):
Train R2: 0.8980

Coefficients:
  x0 :   +0.426
  x1 :   -0.688
  x2 :   -1.890
  x3 :   -2.303
  x4 :   +3.464
  x5 :   -1.441
  x6 :   +1.361
  x7 :   +0.791

In [9]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# Iterative refinement: fit, remove biggest residuals, refit
imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)

print("Testing thresholds for FINAL clean linear model:\n")
for THRESH in [80, 100, 120, 150]:
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    clean_scores = []
    for tr, va in kf.split(X_imp):
        mask_tr = np.abs(y[tr]) <= THRESH
        m = LinearRegression().fit(X_imp[tr][mask_tr], y[tr][mask_tr])
        # score on clean validation only (the "real" signal)
        mask_va = np.abs(y[va]) <= THRESH
        clean_scores.append(r2_score(y[va][mask_va], m.predict(X_imp[va][mask_va])))
    print(f"  THRESH=±{THRESH}: clean CV R2 = {np.mean(clean_scores):.4f}")

Testing thresholds for FINAL clean linear model:

  THRESH=±80: clean CV R2 = 0.9454
  THRESH=±100: clean CV R2 = 0.9568
  THRESH=±120: clean CV R2 = 0.9498
  THRESH=±150: clean CV R2 = 0.9399


In [10]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
import boto3

# ============================================================
# FINAL MODEL: clean training data, fit LinearRegression
# ============================================================
imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
X_test_imp = imp.transform(X_test)

# Remove corrupted giant rows (best clean-R2 threshold)
THRESHOLD = 100
mask = np.abs(y) <= THRESHOLD
print(f"Training on {mask.sum()} clean rows (removed {(~mask).sum()} giants)")

final = LinearRegression().fit(X_imp[mask], y[mask])
print(f"Clean train R2: {final.score(X_imp[mask], y[mask]):.4f}")

# Predict test set
preds = final.predict(X_test_imp)
print(f"Predictions: min={preds.min():.1f}, max={preds.max():.1f}, mean={preds.mean():.1f}")
# ============================================================
# Save submission to S3
# ============================================================
submission = pd.DataFrame({"Id": test_ids, "target": preds})
submission = submission.sort_values("Id").reset_index(drop=True)
print(submission.head())
print("Rows:", len(submission))

submission.to_csv("/tmp/submission.csv", index=False)
s3 = boto3.client("s3")
s3.upload_file("/tmp/submission.csv", "test-magangal", "submission/submission.csv")
print("Saved to s3://test-magangal/submission/submission.csv")

Training on 2123 clean rows (removed 377 giants)
Clean train R2: 0.9578
Predictions: min=-225.6, max=182.4, mean=-0.4
   Id     target
0   0 -57.869576
1   6 -53.150623
2   7 -57.609315
3   8   8.650209
4  12  22.359799
Rows: 2500
Saved to s3://test-magangal/submission/submission.csv


In [11]:
# ============================================================
# FINAL SUBMISSION: Clean Linear Regression
# Technique: remove corrupted giant labels (|y|>100), fit LinearRegression
# Clean CV R2 = 0.957
# ============================================================
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
import boto3

# Impute missing features with median
imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
X_test_imp = imp.transform(X_test)

# Remove corrupted giant rows (the optimal threshold)
THRESHOLD = 100
mask = np.abs(y) <= THRESHOLD
print(f"Clean rows: {mask.sum()} / {len(y)}  (removed {(~mask).sum()} corrupted giants)")

# Fit linear regression on clean data
final = LinearRegression().fit(X_imp[mask], y[mask])
print(f"Clean train R2: {final.score(X_imp[mask], y[mask]):.4f}")

# Predict test set
preds = final.predict(X_test_imp)
print(f"Predictions -> min={preds.min():.1f}, max={preds.max():.1f}, mean={preds.mean():.2f}")

# Build & save submission
submission = pd.DataFrame({"Id": test_ids, "target": preds})
submission = submission.sort_values("Id").reset_index(drop=True)
print(submission.head())
print("Total rows:", len(submission))

submission.to_csv("/tmp/submission.csv", index=False)
s3 = boto3.client("s3")
s3.upload_file("/tmp/submission.csv", "test-magangal", "submission/submission.csv")
print("\n✅ Saved to s3://test-magangal/submission/submission.csv")

Clean rows: 2123 / 2500  (removed 377 corrupted giants)
Clean train R2: 0.9578
Predictions -> min=-225.6, max=182.4, mean=-0.40
   Id     target
0   0 -57.869576
1   6 -53.150623
2   7 -57.609315
3   8   8.650209
4  12  22.359799
Total rows: 2500

✅ Saved to s3://test-magangal/submission/submission.csv


In [12]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
import boto3

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
X_test_imp = imp.transform(X_test)

# Start with loose threshold, then iteratively remove worst residuals
mask = np.abs(y) <= 200
for iteration in range(8):
    m = LinearRegression().fit(X_imp[mask], y[mask])
    resid = np.abs(y - m.predict(X_imp))
    cutoff = np.percentile(resid[mask], 93)   # drop worst 7% each round
    new_mask = resid <= cutoff
    print(f"Iter {iteration}: kept {new_mask.sum()}, cutoff={cutoff:.1f}, "
          f"train R2={m.score(X_imp[new_mask], y[new_mask]):.4f}")
    if new_mask.sum() == mask.sum():
        break
    mask = new_mask

final = LinearRegression().fit(X_imp[mask], y[mask])
print(f"\nFinal clean R2: {final.score(X_imp[mask], y[mask]):.4f}")
print(f"Final rows used: {mask.sum()}")

preds = final.predict(X_test_imp)
submission = pd.DataFrame({"Id": test_ids, "target": preds}).sort_values("Id").reset_index(drop=True)
submission.to_csv("/tmp/submission2.csv", index=False)
s3 = boto3.client("s3")
s3.upload_file("/tmp/submission2.csv", "test-magangal", "submission/submission_iterative.csv")
print("✅ Saved iterative version")

Iter 0: kept 2190, cutoff=9.8, train R2=0.9980
Iter 1: kept 2036, cutoff=1.2, train R2=0.9999
Iter 2: kept 1893, cutoff=0.9, train R2=1.0000
Iter 3: kept 1760, cutoff=0.7, train R2=1.0000
Iter 4: kept 1636, cutoff=0.6, train R2=1.0000
Iter 5: kept 1521, cutoff=0.6, train R2=1.0000
Iter 6: kept 1414, cutoff=0.5, train R2=1.0000
Iter 7: kept 1315, cutoff=0.4, train R2=1.0000

Final clean R2: 1.0000
Final rows used: 1315
✅ Saved iterative version


In [13]:
# ============================================================
# REGRESSION CHALLENGE - FULL SOLUTION
# Insight: Target is an EXACT linear function of features,
#          but ~half the training labels are corrupted (giants).
# Strategy: Iteratively remove corrupted rows -> recover the
#           exact linear coefficients -> predict test set.
# ============================================================

# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import boto3
from pyspark.sql import SparkSession
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

spark = SparkSession.builder.getOrCreate()

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
TEST_PATH  = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_test.csv"
BUCKET     = "test-magangal"
OUTPUT_KEY = "submission/submission.csv"

# ------------------------------------------------------------
# Step 1: Load data
# ------------------------------------------------------------
train = (spark.read.option("header", "true").option("inferSchema", "true")
         .csv(TRAIN_PATH).toPandas())
test  = (spark.read.option("header", "true").option("inferSchema", "true")
         .csv(TEST_PATH).toPandas())

print("Train shape:", train.shape)
print("Test  shape:", test.shape)

# ------------------------------------------------------------
# Step 2: Prepare features (ensure numeric)
# ------------------------------------------------------------
feature_cols = [f"x{i}" for i in range(15)]

for c in feature_cols + ["target"]:
    if c in train.columns:
        train[c] = pd.to_numeric(train[c], errors="coerce")
for c in feature_cols:
    test[c] = pd.to_numeric(test[c], errors="coerce")

X        = train[feature_cols].values
y        = train["target"].values.astype(float)
X_test   = test[feature_cols].values
test_ids = test["Id"].values

# ------------------------------------------------------------
# Step 3: Impute missing feature values (median)
# ------------------------------------------------------------
imp = SimpleImputer(strategy="median")
X_imp      = imp.fit_transform(X)
X_test_imp = imp.transform(X_test)

# ------------------------------------------------------------
# Step 4: Iteratively remove corrupted rows to recover the
#         EXACT linear relationship.
#         Keep only points that lie (almost) exactly on the line.
# ------------------------------------------------------------
mask = np.abs(y) <= 200          # start by dropping the obvious giants

for iteration in range(15):
    m = LinearRegression().fit(X_imp[mask], y[mask])
    resid = np.abs(y - m.predict(X_imp))
    new_mask = resid <= 1.0      # within 1.0 of perfect prediction
    r2 = m.score(X_imp[new_mask], y[new_mask])
    print(f"Iter {iteration}: kept {new_mask.sum():>4} rows, R2 = {r2:.6f}")
    if new_mask.sum() == mask.sum():
        break
    mask = new_mask

# ------------------------------------------------------------
# Step 5: Fit final exact model on clean rows
# ------------------------------------------------------------
final = LinearRegression().fit(X_imp[mask], y[mask])

print(f"\n=== FINAL MODEL ===")
print(f"Clean rows used : {mask.sum()} / {len(y)}")
print(f"Train R2        : {final.score(X_imp[mask], y[mask]):.8f}")
print(f"Max residual    : {np.abs(y[mask] - final.predict(X_imp[mask])).max():.2e}")

print("\nExact coefficients:")
for i in range(15):
    print(f"  x{i:<2}: {final.coef_[i]:+.6f}")
print(f"  intercept: {final.intercept_:+.6f}")

# ------------------------------------------------------------
# Step 6: Predict the test set
# ------------------------------------------------------------
preds = final.predict(X_test_imp)
print(f"\nPredictions -> min={preds.min():.2f}, max={preds.max():.2f}, mean={preds.mean():.2f}")

# ------------------------------------------------------------
# Step 7: Build submission and save to S3
# ------------------------------------------------------------
submission = (pd.DataFrame({"Id": test_ids, "target": preds})
              .sort_values("Id").reset_index(drop=True))

print("\nSubmission preview:")
print(submission.head())
print("Total rows:", len(submission))

# Save locally then upload (single clean CSV)
submission.to_csv("/tmp/submission.csv", index=False)
s3 = boto3.client("s3")
s3.upload_file("/tmp/submission.csv", BUCKET, OUTPUT_KEY)

print(f"\n✅ Saved submission to s3://{BUCKET}/{OUTPUT_KEY}")


Train shape: (2500, 17)
Test  shape: (2500, 16)
Iter 0: kept  695 rows, R2 = 0.999894
Iter 1: kept 1258 rows, R2 = 0.999908
Iter 2: kept 1784 rows, R2 = 0.999925
Iter 3: kept 1950 rows, R2 = 0.999941
Iter 4: kept 1969 rows, R2 = 0.999944
Iter 5: kept 1971 rows, R2 = 0.999944
Iter 6: kept 1972 rows, R2 = 0.999944
Iter 7: kept 1971 rows, R2 = 0.999944
Iter 8: kept 1971 rows, R2 = 0.999944

=== FINAL MODEL ===
Clean rows used : 1971 / 2500
Train R2        : 0.99994373
Max residual    : 9.98e-01

Exact coefficients:
  x0 : +0.350561
  x1 : -0.769054
  x2 : -1.875257
  x3 : -2.174794
  x4 : +3.495165
  x5 : -1.488013
  x6 : +1.273181
  x7 : +0.779198
  x8 : -2.380936
  x9 : +0.846701
  x10: +0.590470
  x11: -0.046806
  x12: -0.862767
  x13: -0.005420
  x14: +0.943384
  intercept: -0.008124

Predictions -> min=-229.30, max=184.77, mean=-0.54

Submission preview:
   Id     target
0   0 -58.612729
1   6 -55.135746
2   7 -59.120823
3   8  10.809109
4  12  22.076025
Total rows: 2500

✅ Saved sub

In [14]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import boto3

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
X_test_imp = imp.transform(X_test)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# Find what scores best on FULL data (giants included)
# ============================================================
def cv_full(make_model):
    scores=[]
    for tr,va in kf.split(X_imp):
        m=make_model().fit(X_imp[tr], y[tr])
        scores.append(r2_score(y[va], m.predict(X_imp[va])))
    return np.mean(scores)

print("Full-data CV R2 (matches leaderboard):")
print(f"  Plain Linear: {cv_full(lambda: LinearRegression()):.5f}")
for a in [0.1, 1, 10, 100, 1000, 5000, 10000, 50000]:
    print(f"  Ridge a={a:>6}: {cv_full(lambda a=a: Ridge(alpha=a)):.5f}")

Full-data CV R2 (matches leaderboard):
  Plain Linear: -0.45738
  Ridge a=   0.1: -0.45737
  Ridge a=     1: -0.45736
  Ridge a=    10: -0.45725
  Ridge a=   100: -0.45615
  Ridge a=  1000: -0.44534
  Ridge a=  5000: -0.40157
  Ridge a= 10000: -0.35480
  Ridge a= 50000: -0.14897


In [15]:
import numpy as np, pandas as pd, boto3
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy="median")
X_imp = imp.fit_transform(X)
X_test_imp = imp.transform(X_test)

final = LinearRegression().fit(X_imp, y)   # NO cleaning, all data
preds = final.predict(X_test_imp)

submission = pd.DataFrame({"Id": test_ids, "target": preds}).sort_values("Id").reset_index(drop=True)
submission.to_csv("/tmp/s.csv", index=False)
boto3.client("s3").upload_file("/tmp/s.csv", "test-magangal", "submission/submission.csv")
print("Saved plain linear. Pred range:", preds.min(), preds.max())

Saved plain linear. Pred range: -1979.047555323056 1556.7422149733138


In [2]:
from pyspark.sql import functions as F

# ------------------------------------------------------------
# Step 1: Load data from S3
# ------------------------------------------------------------
TRAIN_PATH  = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
TEST_PATH   = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_test.csv"
OUTPUT_PATH = "s3://test-magangal/submission/"

train = (spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(TRAIN_PATH))

test = (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(TEST_PATH))

print("Train rows:", train.count(), " cols:", len(train.columns))
print("Test  rows:", test.count(),  " cols:", len(test.columns))
print("Train columns:", train.columns)
print("Test  columns:", test.columns)
train.printSchema()
train.show(5)
# ------------------------------------------------------------
# Step 2: Missing value counts per column
# ------------------------------------------------------------
feature_cols = [f"x{i}" for i in range(15)]

print("=== TRAIN missing counts ===")
train.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in feature_cols + ["target"]
]).show()

print("=== TEST missing counts ===")
test.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in feature_cols
]).show()
# ------------------------------------------------------------
# Step 3: Target distribution + outlier inspection  (CRITICAL)
# ------------------------------------------------------------
train.select(
    F.min("target").alias("min"),
    F.max("target").alias("max"),
    F.mean("target").alias("mean"),
    F.expr("percentile_approx(target, 0.01)").alias("p01"),
    F.expr("percentile_approx(target, 0.05)").alias("p05"),
    F.expr("percentile_approx(target, 0.25)").alias("p25"),
    F.expr("percentile_approx(target, 0.50)").alias("median"),
    F.expr("percentile_approx(target, 0.75)").alias("p75"),
    F.expr("percentile_approx(target, 0.95)").alias("p95"),
    F.expr("percentile_approx(target, 0.99)").alias("p99")
).show()

print("rows |target|>1000 :", train.filter(F.abs(F.col("target")) > 1000).count())
print("rows |target|>200  :", train.filter(F.abs(F.col("target")) > 200).count())
print("rows |target|>100  :", train.filter(F.abs(F.col("target")) > 100).count())
# ------------------------------------------------------------
# Step 4: Linear correlation of each feature with target
# ------------------------------------------------------------
print("=== Pearson correlation (feature vs target) ===")
for c in feature_cols:
    print(f"{c:>4}: {train.stat.corr(c, 'target'):+.4f}")
# ------------------------------------------------------------
# Step 5: Inspect the extreme-target rows — distinguishable in X?
# ------------------------------------------------------------
print("=== Extreme target rows (|target|>1000) ===")
train.filter(F.abs(F.col("target")) > 1000) \
     .select(["Id", "target"] + feature_cols) \
     .show(50, truncate=False)
# ------------------------------------------------------------
# Step 6: Feature scale summary (for scaling decision)
# ------------------------------------------------------------
train.select([
    F.round(F.mean(c), 2).alias(f"{c}_mean") for c in feature_cols
] + [
    F.round(F.stddev(c), 2).alias(f"{c}_std") for c in feature_cols
]).show(truncate=False)

Train rows: 2500  cols: 17
Test  rows: 2500  cols: 16
Train columns: ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'target', 'Id']
Test  columns: ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'Id']
root
 |-- x0: double (nullable = true)
 |-- x1: double (nullable = true)
 |-- x2: double (nullable = true)
 |-- x3: double (nullable = true)
 |-- x4: double (nullable = true)
 |-- x5: double (nullable = true)
 |-- x6: double (nullable = true)
 |-- x7: double (nullable = true)
 |-- x8: double (nullable = true)
 |-- x9: double (nullable = true)
 |-- x10: double (nullable = true)
 |-- x11: double (nullable = true)
 |-- x12: double (nullable = true)
 |-- x13: double (nullable = true)
 |-- x14: double (nullable = true)
 |-- target: double (nullable = true)
 |-- Id: integer (nullable = true)

+-------------------+------------------+------------------+------------------+-------------------+----------

In [3]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Pull to pandas (small data)
# ------------------------------------------------------------
pdf_train = train.toPandas()
pdf_test  = test.toPandas()

feature_cols = [f"x{i}" for i in range(15)]

X      = pdf_train[feature_cols].copy()
y      = pdf_train["target"].values
X_test = pdf_test[feature_cols].copy()
test_ids = pdf_test["Id"].values

print("Train X:", X.shape, " Test X:", X_test.shape)
# ------------------------------------------------------------
# Quick hypothesis test: does degree-2 (interactions) explain target?
# ------------------------------------------------------------
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Model A: plain linear (baseline sanity check)
lin = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
    ("lr",  LinearRegression())
])
print("Linear (deg1)  R2:", cross_val_score(lin, X, y, cv=kf, scoring="r2").mean())

# Model B: degree-2 polynomial (interactions + squares)
poly2 = Pipeline([
    ("imp",  SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("sc",   StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])
print("Poly deg2      R2:", cross_val_score(poly2, X, y, cv=kf, scoring="r2").mean())

# Model C: interaction-only (no squares) — often the true generator
inter = Pipeline([
    ("imp",  SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ("sc",   StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])
print("Interaction    R2:", cross_val_score(inter, X, y, cv=kf, scoring="r2").mean())

Train X: (2500, 15)  Test X: (2500, 15)
Linear (deg1)  R2: -0.45720536806316847
Poly deg2      R2: -2.1199922085769467
Interaction    R2: -2.0077064142422274


In [4]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

imp = SimpleImputer(strategy="median")
Xi = imp.fit_transform(X)

poly = PolynomialFeatures(degree=2, include_bias=False)
Xp = poly.fit_transform(Xi)
sc = StandardScaler().fit(Xp)
Xs = sc.transform(Xp)

# Out-of-fold predictions
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(y))
for tr, va in kf.split(Xs):
    m = Ridge(alpha=10.0).fit(Xs[tr], y[tr])
    oof[va] = m.predict(Xs[va])

big = np.abs(y) > 1000
small = ~big

print("Overall R2          :", r2_score(y, oof))
print("R2 on SMALL targets :", r2_score(y[small], oof[small]))
print("R2 on BIG targets   :", r2_score(y[big],   oof[big]))
print()
# Show a few big-target predictions vs truth
for i in np.where(big)[0][:15]:
    print(f"true={y[i]:>12.1f}   pred={oof[i]:>12.1f}")
# Also: correlation between |target| and product magnitudes
# Test if target relates to pairwise products specifically
prod_max = np.max(np.abs(Xi[:, :, None] * Xi[:, None, :]), axis=(1,2))
print("corr(|target|, max|xi*xj|):", np.corrcoef(np.abs(y), prod_max)[0,1])

Overall R2          : -0.16788786977115322
R2 on SMALL targets : -71.14486421867164
R2 on BIG targets   : 0.02712175627643698

true=     -2123.1   pred=     -1182.0
true=     -1556.0   pred=       619.0
true=     -3877.3   pred=      -716.9
true=     -1989.9   pred=     -1618.8
true=     -8400.0   pred=     -3282.1
true=     -5475.7   pred=     -1809.2
true=     -2484.5   pred=         6.2
true=      1052.1   pred=       176.6
true=     -2063.0   pred=       316.2
true=      1561.7   pred=      2406.1
true=      3539.8   pred=      2504.2
true=     -3062.5   pred=      -655.9
true=     -1853.9   pred=     -1404.2
true=     13608.8   pred=       415.1
true=      9529.8   pred=       262.7
corr(|target|, max|xi*xj|): 0.045227894205172876


In [5]:
import numpy as np
from itertools import combinations
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score
from scipy.stats import pearsonr, spearmanr

feature_cols = [f"x{i}" for i in range(15)]
imp = SimpleImputer(strategy="median")
Xi  = imp.fit_transform(X)

# Work on clean core so outliers don't mask signal
mask = np.abs(y) <= 200
Xc, yc = Xi[mask], y[mask]
print(f"Clean rows: {mask.sum()}")
# ---- 1) Each individual PAIRWISE PRODUCT correlation with target ----
print("=== Top individual products xi*xj ===")
res = []
for i, j in combinations(range(15), 2):
    prod = Xc[:, i] * Xc[:, j]
    r = pearsonr(prod, yc)[0]
    res.append((abs(r), r, i, j))
res.sort(reverse=True)
for ar, r, i, j in res[:10]:
    print(f"x{i}*x{j}: r={r:+.3f}")
# ---- 2) Each SQUARE term ----
print("\n=== Squares xi^2 ===")
for i in range(15):
    r = pearsonr(Xc[:, i]**2, yc)[0]
    if abs(r) > 0.1:
        print(f"x{i}^2: r={r:+.3f}")

# ---- 3) RATIOS xi/xj (guard divide-by-zero) ----
print("\n=== Top ratios xi/xj ===")
res = []
for i in range(15):
    for j in range(15):
        if i == j: continue
        denom = Xc[:, j]
        safe = np.abs(denom) > 0.5     # avoid blow-ups
        if safe.sum() < 100: continue
        ratio = Xc[safe, i] / denom[safe]
        r = pearsonr(ratio, yc[safe])[0]
        res.append((abs(r), r, i, j))
res.sort(reverse=True)
for ar, r, i, j in res[:10]:
    print(f"x{i}/x{j}: r={r:+.3f}")
# ---- 4) SPEARMAN (catches any monotonic nonlinear single-feature link) ----
print("\n=== Spearman (single features) ===")
for i in range(15):
    rho = spearmanr(Xc[:, i], yc)[0]
    print(f"x{i}: spearman={rho:+.3f}")
# ---- 5) Let TREES find arbitrary combinations automatically ----
kf = KFold(5, shuffle=True, random_state=42)

rf = RandomForestRegressor(n_estimators=300, max_depth=None, n_jobs=-1, random_state=42)
print("\nRandomForest (clean) R2:", cross_val_score(rf, Xc, yc, cv=kf, scoring="r2").mean())

gb = GradientBoostingRegressor(n_estimators=400, max_depth=3, learning_rate=0.05, random_state=42)
print("GradientBoosting (clean) R2:", cross_val_score(gb, Xc, yc, cv=kf, scoring="r2").mean())

# Feature importances from RF (which combos it relied on)
rf.fit(Xc, yc)
imp_sorted = sorted(zip(feature_cols, rf.feature_importances_), key=lambda t:-t[1])
print("\nRF importances:", [(f, round(v,3)) for f,v in imp_sorted])

Clean rows: 2354
=== Top individual products xi*xj ===
x6*x7: r=-0.065
x2*x5: r=+0.061
x4*x9: r=-0.057
x0*x2: r=-0.055
x2*x10: r=+0.048
x5*x11: r=+0.048
x12*x13: r=+0.044
x8*x9: r=+0.044
x4*x8: r=+0.043
x6*x8: r=-0.037

=== Squares xi^2 ===

=== Top ratios xi/xj ===
x4/x1: r=-0.068
x0/x6: r=+0.058
x6/x1: r=-0.057
x2/x5: r=+0.057
x7/x2: r=-0.054
x2/x13: r=-0.054
x10/x6: r=-0.053
x7/x12: r=+0.053
x3/x12: r=+0.052
x6/x8: r=-0.051

=== Spearman (single features) ===
x0: spearman=+0.045
x1: spearman=-0.155
x2: spearman=-0.229
x3: spearman=-0.060
x4: spearman=+0.740
x5: spearman=-0.343
x6: spearman=+0.149
x7: spearman=+0.113
x8: spearman=-0.273
x9: spearman=+0.081
x10: spearman=+0.090
x11: spearman=-0.036
x12: spearman=-0.037
x13: spearman=-0.002
x14: spearman=+0.046

RandomForest (clean) R2: 0.7962891098588797
GradientBoosting (clean) R2: 0.8599171387061588

RF importances: [('x4', 0.568), ('x5', 0.139), ('x8', 0.077), ('x2', 0.057), ('x10', 0.023), ('x7', 0.023), ('x1', 0.021), ('x6', 0.02

In [6]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score

feature_cols = [f"x{i}" for i in range(15)]
imp = SimpleImputer(strategy="median")
Xi  = imp.fit_transform(X)

kf = KFold(5, shuffle=True, random_state=42)

# 1) Confirm best trim threshold
print("=== Trim threshold sweep (GradientBoosting) ===")
for THRESH in [100, 150, 200, 300, 500, 1000]:
    m = np.abs(y) <= THRESH
    gb = GradientBoostingRegressor(n_estimators=500, max_depth=3,
                                   learning_rate=0.03, subsample=0.8, random_state=42)
    r2 = cross_val_score(gb, Xi[m], y[m], cv=kf, scoring="r2").mean()
    print(f"THRESH={THRESH:>4}  kept={m.sum():>4}  R2={r2:+.4f}")
# 2) Compare GBM variants at the chosen threshold (use whichever threshold won above)
THRESH = 200
m = np.abs(y) <= THRESH
Xc, yc = Xi[m], y[m]

models = {
    "GBM depth3 n500 lr03": GradientBoostingRegressor(n_estimators=500, max_depth=3, learning_rate=0.03, subsample=0.8, random_state=42),
    "GBM depth4 n800 lr02": GradientBoostingRegressor(n_estimators=800, max_depth=4, learning_rate=0.02, subsample=0.8, random_state=42),
    "HistGBM":              HistGradientBoostingRegressor(max_iter=800, learning_rate=0.03, max_depth=None, l2_regularization=1.0, random_state=42),
}
for name, mdl in models.items():
    r2 = cross_val_score(mdl, Xc, yc, cv=kf, scoring="r2").mean()
    print(f"{name:>22}: R2={r2:+.4f}")
# 3) Examine the x4 -> target relationship to understand its shape
import numpy as np
order = np.argsort(Xc[:, 4])
x4s = Xc[order, 4]; ys = yc[order]
# bucketed means
bins = np.linspace(x4s.min(), x4s.max(), 11)
idx = np.digitize(x4s, bins)
print("=== mean(target) by x4 bucket ===")
for b in range(1, 11):
    sel = idx == b
    if sel.sum() > 0:
        print(f"x4 in [{bins[b-1]:+.1f},{bins[b]:+.1f}]  mean_target={ys[sel].mean():+8.2f}  n={sel.sum()}")

=== Trim threshold sweep (GradientBoosting) ===
THRESH= 100  kept=2123  R2=+0.9061
THRESH= 150  kept=2317  R2=+0.9057
THRESH= 200  kept=2354  R2=+0.8627
THRESH= 300  kept=2373  R2=+0.7933
THRESH= 500  kept=2387  R2=+0.6571
THRESH=1000  kept=2416  R2=+0.2930
  GBM depth3 n500 lr03: R2=+0.8627
  GBM depth4 n800 lr02: R2=+0.8622
               HistGBM: R2=+0.8520
=== mean(target) by x4 bucket ===
x4 in [-43.4,-34.8]  mean_target= -159.43  n=5
x4 in [-34.8,-26.2]  mean_target= -100.12  n=40
x4 in [-26.2,-17.6]  mean_target=  -74.27  n=131
x4 in [-17.6,-9.0]  mean_target=  -43.18  n=380
x4 in [-9.0,-0.4]  mean_target=  -15.69  n=645
x4 in [-0.4,+8.2]  mean_target=  +14.86  n=552
x4 in [+8.2,+16.8]  mean_target=  +44.19  n=364
x4 in [+16.8,+25.4]  mean_target=  +70.54  n=183
x4 in [+25.4,+34.1]  mean_target=  +92.59  n=40
x4 in [+34.1,+42.7]  mean_target= +119.46  n=13


In [7]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, HuberRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

feature_cols = [f"x{i}" for i in range(15)]
imp = SimpleImputer(strategy="median")
Xi  = imp.fit_transform(X)

kf = KFold(5, shuffle=True, random_state=42)

for THRESH in [100, 150]:
    m = np.abs(y) <= THRESH
    Xc, yc = Xi[m], y[m]
    print(f"\n===== THRESH={THRESH}  (kept {m.sum()}) =====")

    lin = Pipeline([("sc", StandardScaler()), ("lr", LinearRegression())])
    print("  Linear      :", round(cross_val_score(lin, Xc, yc, cv=kf, scoring='r2').mean(), 4))

    ridge = Pipeline([("sc", StandardScaler()), ("r", Ridge(alpha=1.0))])
    print("  Ridge       :", round(cross_val_score(ridge, Xc, yc, cv=kf, scoring='r2').mean(), 4))

    huber = Pipeline([("sc", StandardScaler()), ("h", HuberRegressor(max_iter=3000))])
    print("  Huber       :", round(cross_val_score(huber, Xc, yc, cv=kf, scoring='r2').mean(), 4))

    gbm = GradientBoostingRegressor(n_estimators=500, max_depth=3, learning_rate=0.03,
                                    subsample=0.8, random_state=42)
    print("  GBM         :", round(cross_val_score(gbm, Xc, yc, cv=kf, scoring='r2').mean(), 4))
# Show the linear coefficients — see which features matter and the slope on x4
m = np.abs(y) <= 150
Xc, yc = Xi[m], y[m]
sc = StandardScaler().fit(Xc)
lr = LinearRegression().fit(sc.transform(Xc), yc)
coefs = sorted(zip(feature_cols, lr.coef_), key=lambda t: -abs(t[1]))
print("Linear coefficients (standardized), sorted by importance:")
for f, c in coefs:
    print(f"  {f:>4}: {c:+8.2f}")


===== THRESH=100  (kept 2123) =====
  Linear      : 0.9572
  Ridge       : 0.9572
  Huber       : 0.9571
  GBM         : 0.9061

===== THRESH=150  (kept 2317) =====
  Linear      : 0.9399
  Ridge       : 0.9399
  Huber       : 0.9402
  GBM         : 0.9057
Linear coefficients (standardized), sorted by importance:
    x4:   +42.72
    x5:   -22.20
    x8:   -16.19
    x2:   -14.75
    x1:    -9.12
   x10:    +7.86
    x7:    +7.75
    x6:    +6.93
    x9:    +6.49
    x3:    -4.49
   x14:    +3.29
    x0:    +3.19
   x12:    -2.80
   x11:    -0.52
   x13:    -0.11


In [3]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# ------------------------------------------------------------
# 1. Prepare data (already in pandas: X, y, X_test, test_ids)
# ------------------------------------------------------------
feature_cols = [f"x{i}" for i in range(15)]

X_full   = pdf_train[feature_cols].values
y_full   = pdf_train["target"].values
X_test_v = pdf_test[feature_cols].values
test_ids = pdf_test["Id"].values

# ------------------------------------------------------------
# 2. Impute missing values (median, fit on TRAIN only)
# ------------------------------------------------------------
imp = SimpleImputer(strategy="median")
X_full_i = imp.fit_transform(X_full)
X_test_i = imp.transform(X_test_v)      # use TRAIN medians on test

# ------------------------------------------------------------
# 3. Trim corrupted target rows  (choose THRESH)
# ------------------------------------------------------------
THRESH = 100
mask = np.abs(y_full) <= THRESH
X_clean, y_clean = X_full_i[mask], y_full[mask]
print(f"Training on {mask.sum()} clean rows (dropped {len(y_full)-mask.sum()})")

# ------------------------------------------------------------
# 4. Scale + fit Linear Regression
# ------------------------------------------------------------
scaler = StandardScaler().fit(X_clean)
X_clean_s = scaler.transform(X_clean)
X_test_s  = scaler.transform(X_test_i)

model = LinearRegression().fit(X_clean_s, y_clean)
print("Train R2 on clean data:", round(model.score(X_clean_s, y_clean), 4))

# ------------------------------------------------------------
# 5. Predict on test
# ------------------------------------------------------------
preds = model.predict(X_test_s)

# Optional: clip to clean range (test BOTH versions via submission)
preds_clipped = np.clip(preds, -THRESH, THRESH)

# ------------------------------------------------------------
# 6. Write submission to S3
# ------------------------------------------------------------
submission = pd.DataFrame({"Id": test_ids, "target": preds})   # try unclipped first
sub_spark = spark.createDataFrame(submission)

(sub_spark.coalesce(1)
    .write.mode("overwrite")
    .option("header", "true")
    .csv("s3://test-magangal/submission/"))

print("Submission written. Sample:")
print(submission.head())

NameError: name 'pdf_train' is not defined


In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score

# ============================================================
# 0a. Reload from S3 (in case Spark df's are also gone)
# ============================================================
TRAIN_PATH = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_train.csv"
TEST_PATH  = "s3://test-magangal/spring2026_kaggle_linear_regression_challenge_test.csv"

train = (spark.read.option("header", "true").option("inferSchema", "true").csv(TRAIN_PATH))
test  = (spark.read.option("header", "true").option("inferSchema", "true").csv(TEST_PATH))

# ============================================================
# 0b. Convert Spark -> pandas  (THIS creates pdf_train)
# ============================================================
pdf_train = train.toPandas()
pdf_test  = test.toPandas()

print("pdf_train:", pdf_train.shape)
print("pdf_test :", pdf_test.shape)
print("columns  :", pdf_train.columns.tolist())
# ============================================================
# 1. Build X / y arrays
# ============================================================
feature_cols = [f"x{i}" for i in range(15)]

X_all = pdf_train[feature_cols].values
y_all = pdf_train["target"].values

print("X_all:", X_all.shape, " y range:", y_all.min(), "to", y_all.max())
# ============================================================
# 2. Train / validation split
# ============================================================
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, shuffle=True
)
print("Train:", X_train.shape, " Val:", X_val.shape)
# ============================================================
# 3. Impute (fit on train only)
# ============================================================
imp = SimpleImputer(strategy="median")
X_train_i = imp.fit_transform(X_train)
X_val_i   = imp.transform(X_val)
# ============================================================
# 4. MODEL A: Trimmed Linear  (train clean, eval on full val)
# ============================================================
THRESH = 100
mask = np.abs(y_train) <= THRESH
X_train_c, y_train_c = X_train_i[mask], y_train[mask]

scaler = StandardScaler().fit(X_train_c)
modelA = LinearRegression().fit(scaler.transform(X_train_c), y_train_c)
predA  = modelA.predict(scaler.transform(X_val_i))

r2A_full  = r2_score(y_val, predA)                                  # honest (like Kaggle)
clean_val = np.abs(y_val) <= THRESH
r2A_clean = r2_score(y_val[clean_val], predA[clean_val])            # optimistic

print("MODEL A (trimmed linear)")
print("  R2 on FULL val (HONEST) :", round(r2A_full, 4))
print("  R2 on CLEAN val only    :", round(r2A_clean, 4))
# ============================================================
# 5. MODEL B: log-transform target, train on ALL data
# ============================================================
def fwd(t): return np.sign(t) * np.log1p(np.abs(t))
def inv(t): return np.sign(t) * np.expm1(np.abs(t))

scalerB = StandardScaler().fit(X_train_i)
modelB  = Ridge(alpha=1.0).fit(scalerB.transform(X_train_i), fwd(y_train))
predB   = inv(modelB.predict(scalerB.transform(X_val_i)))

r2B_full = r2_score(y_val, predB)
print("MODEL B (log-transform, all data)")
print("  R2 on FULL val (HONEST) :", round(r2B_full, 4))
# ============================================================
# 6. Comparison
# ============================================================
print("\n===== HONEST COMPARISON (full val = Kaggle-like) =====")
print("A) Trimmed Linear      :", round(r2A_full, 5))
print("B) Log-transform Ridge :", round(r2B_full, 4))

pdf_train: (2500, 17)
pdf_test : (2500, 16)
columns  : ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'target', 'Id']
X_all: (2500, 15)  y range: -41008.015927026514 to 69628.19987395463
Train: (2000, 15)  Val: (500, 15)
MODEL A (trimmed linear)
  R2 on FULL val (HONEST) : 0.0053
  R2 on CLEAN val only    : 0.9424
MODEL B (log-transform, all data)
  R2 on FULL val (HONEST) : -0.5245

===== HONEST COMPARISON (full val = Kaggle-like) =====
A) Trimmed Linear      : 0.00534
B) Log-transform Ridge : -0.5245


In [8]:
import numpy as np
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score

# imputed train/val already exist: X_train_i, X_val_i, y_train, y_val

scaler = StandardScaler().fit(X_train_i)
Xtr = scaler.transform(X_train_i)
Xva = scaler.transform(X_val_i)

print("=== ALL DATA, NO TRIM (this is the real game) ===")

# Plain OLS
m = LinearRegression().fit(Xtr, y_train)
print("Plain OLS         :", round(r2_score(y_val, m.predict(Xva)), 5))

# Ridge sweep
for a in [0.01, 0.1, 1, 10, 100, 1000, 10000]:
    m = Ridge(alpha=a).fit(Xtr, y_train)
    print(f"Ridge alpha={a:<7}: {r2_score(y_val, m.predict(Xva)):+.5f}")

# Polynomial degree 2 on all data
poly = PolynomialFeatures(degree=2, include_bias=False)
Xtr2 = StandardScaler().fit_transform(poly.fit_transform(X_train_i))
sc2  = StandardScaler().fit(poly.fit_transform(X_train_i))
Xtr2 = sc2.transform(poly.transform(X_train_i))
Xva2 = sc2.transform(poly.transform(X_val_i))
print("\n=== Poly degree 2, all data ===")
for a in [1, 10, 100, 1000, 10000]:
    m = Ridge(alpha=a).fit(Xtr2, y_train)
    print(f"Poly2 Ridge alpha={a:<7}: {r2_score(y_val, m.predict(Xva2)):+.5f}")

=== ALL DATA, NO TRIM (this is the real game) ===
Plain OLS         : 0.05872
Ridge alpha=0.01   : +0.05872
Ridge alpha=0.1    : +0.05872
Ridge alpha=1      : +0.05872
Ridge alpha=10     : +0.05865
Ridge alpha=100    : +0.05793
Ridge alpha=1000   : +0.04872
Ridge alpha=10000  : +0.01503

=== Poly degree 2, all data ===
Poly2 Ridge alpha=1      : -0.16544
Poly2 Ridge alpha=10     : -0.16399
Poly2 Ridge alpha=100    : -0.15056
Poly2 Ridge alpha=1000   : -0.07810
Poly2 Ridge alpha=10000  : -0.00888


In [9]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

feature_cols = [f"x{i}" for i in range(15)]

# ---- Full training data (NO trimming!) ----
X_full   = pdf_train[feature_cols].values
y_full   = pdf_train["target"].values
X_test_v = pdf_test[feature_cols].values
test_ids = pdf_test["Id"].values

# ---- Impute (fit on train, apply to test) ----
imp = SimpleImputer(strategy="median")
X_full_i = imp.fit_transform(X_full)
X_test_i = imp.transform(X_test_v)

# ---- Scale ----
scaler = StandardScaler().fit(X_full_i)
X_full_s = scaler.transform(X_full_i)
X_test_s = scaler.transform(X_test_i)

# ---- Train plain OLS on ALL data ----
model = LinearRegression().fit(X_full_s, y_full)
print("Train R2 (all data):", round(model.score(X_full_s, y_full), 5))

# ---- Predict test (NO clipping — outliers need full-range predictions) ----
preds = model.predict(X_test_s)

# ---- Build clean submission with pandas (NOT spark) ----
submission = pd.DataFrame({"Id": test_ids, "target": preds}).sort_values("Id")

# Sanity checks
assert submission.shape[0] == 2500
assert submission["target"].isna().sum() == 0
print(submission.head())
print("pred range:", submission["target"].min(), "to", submission["target"].max())

# ---- Save locally then push a SINGLE clean csv to S3 ----
submission.to_csv("/tmp/submission.csv", index=False)

import boto3
boto3.client("s3").upload_file(
    "/tmp/submission.csv", "test-magangal", "submission/submission.csv"
)
print("Uploaded to s3://test-magangal/submission/submission.csv")

Train R2 (all data): 0.06201
   Id      target
0   0 -547.770211
1   6 -426.196767
2   7    6.481453
3   8 -549.065862
4  12  666.589235
pred range: -1979.0475553230553 to 1556.7422149733131
Uploaded to s3://test-magangal/submission/submission.csv


In [10]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score

feature_cols = [f"x{i}" for i in range(15)]
X_all = pdf_train[feature_cols].values
y_all = pdf_train["target"].values

print("R2 across different random splits (shows instability):")
scores = []
for seed in range(20):
    Xtr, Xva, ytr, yva = train_test_split(X_all, y_all, test_size=0.2, random_state=seed)
    imp = SimpleImputer(strategy="median")
    Xtr_i, Xva_i = imp.fit_transform(Xtr), imp.transform(Xva)
    sc = StandardScaler().fit(Xtr_i)
    m = LinearRegression().fit(sc.transform(Xtr_i), ytr)
    r2 = r2_score(yva, m.predict(sc.transform(Xva_i)))
    scores.append(r2)
    print(f"  seed {seed:>2}: {r2:+.4f}")

print(f"\nMean: {np.mean(scores):+.4f}   Std: {np.std(scores):.4f}")
print(f"Min:  {np.min(scores):+.4f}   Max: {np.max(scores):+.4f}")

R2 across different random splits (shows instability):
  seed  0: +0.0585
  seed  1: +0.0459
  seed  2: +0.0611
  seed  3: +0.0442
  seed  4: -1.2628
  seed  5: +0.0300
  seed  6: -0.5451
  seed  7: +0.0398
  seed  8: +0.0665
  seed  9: +0.0421
  seed 10: -0.1016
  seed 11: -0.1288
  seed 12: +0.0342
  seed 13: -0.1676
  seed 14: -0.1807
  seed 15: +0.0404
  seed 16: +0.0801
  seed 17: -0.0281
  seed 18: -0.0537
  seed 19: +0.0733

Mean: -0.0926   Std: 0.3038
Min:  -1.2628   Max: +0.0801


In [11]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, HuberRegressor
from sklearn.metrics import r2_score

feature_cols = [f"x{i}" for i in range(15)]
X_all = pdf_train[feature_cols].values
y_all = pdf_train["target"].values

def eval_model(make_model, n=30):
    s = []
    for seed in range(n):
        Xtr, Xva, ytr, yva = train_test_split(X_all, y_all, test_size=0.2, random_state=seed)
        imp = SimpleImputer(strategy="median")
        Xtr_i, Xva_i = imp.fit_transform(Xtr), imp.transform(Xva)
        sc = StandardScaler().fit(Xtr_i)
        m = make_model().fit(sc.transform(Xtr_i), ytr)
        s.append(r2_score(yva, m.predict(sc.transform(Xva_i))))
    return np.mean(s), np.std(s), np.min(s)

print(f"{'Model':<22}{'mean':>9}{'std':>9}{'min':>9}")
configs = [
    ("OLS",              lambda: LinearRegression()),
    ("Ridge a=100",      lambda: Ridge(alpha=100)),
    ("Ridge a=1000",     lambda: Ridge(alpha=1000)),
    ("Ridge a=5000",     lambda: Ridge(alpha=5000)),
    ("Ridge a=20000",    lambda: Ridge(alpha=20000)),
    ("Ridge a=100000",   lambda: Ridge(alpha=100000)),
    ("Huber",            lambda: HuberRegressor(alpha=0.0001, max_iter=5000)),
]
for name, mk in configs:
    mean, std, mn = eval_model(mk)
    print(f"{name:<22}{mean:+9.4f}{std:9.4f}{mn:+9.4f}")
# Also test the ZERO baseline: predict constant = training mean
# (guarantees R2 = 0 on the eval set by construction... almost)
s = []
for seed in range(30):
    Xtr, Xva, ytr, yva = train_test_split(X_all, y_all, test_size=0.2, random_state=seed)
    pred_const = np.full_like(yva, ytr.mean())
    s.append(r2_score(yva, pred_const))
print(f"\nConstant=train mean : mean={np.mean(s):+.4f} std={np.std(s):.4f}")

# And predict constant = 0
s = []
for seed in range(30):
    Xtr, Xva, ytr, yva = train_test_split(X_all, y_all, test_size=0.2, random_state=seed)
    s.append(r2_score(yva, np.zeros_like(yva)))
print(f"Constant=0          : mean={np.mean(s):+.4f} std={np.std(s):.4f}")

Model                      mean      std      min
OLS                     -0.0618   0.2561  -1.2628
Ridge a=100             -0.0466   0.2274  -1.1158
Ridge a=1000            +0.0181   0.0918  -0.4207
Ridge a=5000            +0.0350   0.0144  +0.0083
Ridge a=20000           +0.0139   0.0094  -0.0020
Ridge a=100000          +0.0011   0.0043  -0.0108
Huber                   +0.0013   0.0049  -0.0090

Constant=train mean : mean=-0.0028 std=0.0032
Constant=0          : mean=-0.0029 std=0.0030


In [12]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

feature_cols = [f"x{i}" for i in range(15)]

# ---- Full training data (NO trimming) ----
X_full   = pdf_train[feature_cols].values
y_full   = pdf_train["target"].values
X_test_v = pdf_test[feature_cols].values
test_ids = pdf_test["Id"].values

# ---- Impute (fit on train, apply to test) ----
imp = SimpleImputer(strategy="median")
X_full_i = imp.fit_transform(X_full)
X_test_i = imp.transform(X_test_v)

# ---- Scale ----
scaler   = StandardScaler().fit(X_full_i)
X_full_s = scaler.transform(X_full_i)
X_test_s = scaler.transform(X_test_i)

# ---- Train robust Ridge (alpha=5000) on ALL data ----
model = Ridge(alpha=5000).fit(X_full_s, y_full)

# ---- Predict test ----
preds = model.predict(X_test_s)

# ---- Build clean submission with pandas ----
submission = pd.DataFrame({"Id": test_ids, "target": preds}).sort_values("Id")

# Sanity checks
assert submission.shape[0] == 2500
assert submission["target"].isna().sum() == 0
print(submission.head())
print("pred range:", round(submission["target"].min(),2), "to", round(submission["target"].max(),2))

# ---- Save single clean CSV and upload to S3 ----
submission.to_csv("/tmp/submission.csv", index=False)
import boto3
boto3.client("s3").upload_file(
    "/tmp/submission.csv", "test-magangal", "submission/submission.csv"
)
print("Uploaded to s3://test-magangal/submission/submission.csv")

   Id      target
0   0 -186.827584
1   6 -155.437908
2   7   -1.834802
3   8 -199.459538
4  12  215.209914
pred range: -668.18 to 503.58
Uploaded to s3://test-magangal/submission/submission.csv


In [13]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

feature_cols = [f"x{i}" for i in range(15)]
X_all = pdf_train[feature_cols].values
y_all = pdf_train["target"].values

def stability(alpha, cols=None, n=40):
    s = []
    idx = cols if cols is not None else list(range(15))
    for seed in range(n):
        Xtr, Xva, ytr, yva = train_test_split(X_all[:, idx], y_all, test_size=0.2, random_state=seed)
        imp = SimpleImputer(strategy="median")
        Xtr_i, Xva_i = imp.fit_transform(Xtr), imp.transform(Xva)
        sc = StandardScaler().fit(Xtr_i)
        m = Ridge(alpha=alpha).fit(sc.transform(Xtr_i), ytr)
        s.append(r2_score(yva, m.predict(sc.transform(Xva_i))))
    return np.mean(s), np.std(s), np.min(s)

print("=== Fine alpha sweep (all 15 features) ===")
for a in [3000, 4000, 4500, 5000, 5500, 6000, 7000, 8000, 10000]:
    mean, std, mn = stability(a)
    print(f"alpha={a:>6}: mean={mean:+.4f}  std={std:.4f}  min={mn:+.4f}")
# === Drop near-zero coefficient features ===
# From earlier: x11 (-0.52), x13 (-0.11) were essentially noise
drop_sets = {
    "all 15"        : list(range(15)),
    "drop x13"      : [i for i in range(15) if i != 13],
    "drop x11,x13"  : [i for i in range(15) if i not in (11, 13)],
    "drop x11,x12,x13": [i for i in range(15) if i not in (11, 12, 13)],
}
print("\n=== Feature subset (alpha=5000) ===")
for name, cols in drop_sets.items():
    mean, std, mn = stability(5000, cols=cols)
    print(f"{name:<18}: mean={mean:+.4f}  std={std:.4f}  min={mn:+.4f}")

=== Fine alpha sweep (all 15 features) ===
alpha=  3000: mean=+0.0392  std=0.0218  min=-0.0539
alpha=  4000: mean=+0.0378  std=0.0164  min=-0.0039
alpha=  4500: mean=+0.0367  std=0.0152  min=+0.0094
alpha=  5000: mean=+0.0355  std=0.0146  min=+0.0083
alpha=  5500: mean=+0.0342  std=0.0142  min=+0.0073
alpha=  6000: mean=+0.0330  std=0.0139  min=+0.0065
alpha=  7000: mean=+0.0306  std=0.0134  min=+0.0050
alpha=  8000: mean=+0.0284  std=0.0130  min=+0.0039
alpha= 10000: mean=+0.0246  std=0.0122  min=+0.0021

=== Feature subset (alpha=5000) ===
all 15            : mean=+0.0355  std=0.0146  min=+0.0083
drop x13          : mean=+0.0356  std=0.0146  min=+0.0083
drop x11,x13      : mean=+0.0378  std=0.0157  min=+0.0090
drop x11,x12,x13  : mean=+0.0380  std=0.0155  min=+0.0091


In [14]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

feature_cols = [f"x{i}" for i in range(15)]
X_all = pdf_train[feature_cols].values
y_all = pdf_train["target"].values

def stability(alpha, cols, n=40):
    s = []
    for seed in range(n):
        Xtr, Xva, ytr, yva = train_test_split(X_all[:, cols], y_all, test_size=0.2, random_state=seed)
        imp = SimpleImputer(strategy="median")
        sc = StandardScaler().fit(imp.fit_transform(Xtr))
        m = Ridge(alpha=alpha).fit(sc.transform(imp.transform(Xtr)), ytr)
        s.append(r2_score(yva, m.predict(sc.transform(imp.transform(Xva)))))
    return np.mean(s), np.std(s), np.min(s)

cols = [i for i in range(15) if i not in (11, 12, 13)]
print("=== drop x11,x12,x13 — alpha sweep ===")
for a in [3000, 3500, 4000, 4500, 5000]:
    mean, std, mn = stability(a, cols)
    print(f"alpha={a:>5}: mean={mean:+.4f}  std={std:.4f}  min={mn:+.4f}")

=== drop x11,x12,x13 — alpha sweep ===
alpha= 3000: mean=+0.0431  std=0.0217  min=-0.0435
alpha= 3500: mean=+0.0422  std=0.0188  min=-0.0160
alpha= 4000: mean=+0.0409  std=0.0171  min=+0.0033
alpha= 4500: mean=+0.0395  std=0.0161  min=+0.0103
alpha= 5000: mean=+0.0380  std=0.0155  min=+0.0091


In [15]:
import pandas as pd, boto3
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

def make_submission(alpha, drop_cols, name):
    feature_cols = [f"x{i}" for i in range(15)]
    sel = [c for i, c in enumerate(feature_cols) if i not in drop_cols]

    X_full   = pdf_train[sel].values
    y_full   = pdf_train["target"].values
    X_test_v = pdf_test[sel].values
    test_ids = pdf_test["Id"].values

    imp = SimpleImputer(strategy="median")
    X_full_i = imp.fit_transform(X_full)
    X_test_i = imp.transform(X_test_v)

    scaler = StandardScaler().fit(X_full_i)
    model  = Ridge(alpha=alpha).fit(scaler.transform(X_full_i), y_full)
    preds  = model.predict(scaler.transform(X_test_i))

    sub = pd.DataFrame({"Id": test_ids, "target": preds}).sort_values("Id")
    assert sub.shape[0] == 2500 and sub["target"].isna().sum() == 0
    sub.to_csv("/tmp/submission.csv", index=False)
    boto3.client("s3").upload_file("/tmp/submission.csv", "test-magangal", "submission/submission.csv")
    print(f"[{name}] alpha={alpha}, dropped={drop_cols} -> uploaded ✓")
    print(sub.head(3))

# ---- SUBMISSION 1 ----
make_submission(alpha=4000, drop_cols=(11, 12, 13), name="aggressive")

[aggressive] alpha=4000, dropped=(11, 12, 13) -> uploaded ✓
   Id      target
0   0 -232.879102
1   6 -200.129533
2   7   -8.364132


In [16]:
# ---- FINAL SUBMISSION (last one today): push alpha down a notch ----
make_submission(alpha=3500, drop_cols=(11, 12, 13), name="push-3500")

[push-3500] alpha=3500, dropped=(11, 12, 13) -> uploaded ✓
   Id      target
0   0 -251.035319
1   6 -215.059377
2   7   -7.798558
